# NB04 — CardioMamba-Net

**Project:** CardioMamba-Net · **Stage:** 4 of 5
`01_verify` → `02_preprocess` → `03_baselines` → **`04_cardiomamba_train`** → `05_evaluate`

---

## The model, and why each piece is there

Every component answers a specific weakness identified in `00_admin/PLAN.md` §4. Nothing is here
because it is fashionable.

| | Contribution | Answers | What it does |
|---|---|---|---|
| **C1** | Physics-informed 8-channel input | G2 | The baseline collapses (I, Q) into one demodulated channel by hand. We feed all eight — I, Q, phase, displacement, velocity, **acceleration**, envelope, cardiac residual — through a learnable 1×1 mix, so the network can rediscover arctangent demodulation if that is genuinely optimal. Acceleration is the SCG analogue where the QRS-synchronous mechanical event actually lives. |
| **C2** | Dual-domain encoder | G3 | MultiRes convolution blocks (strong local morphology) running **alongside** a learnable lifting-wavelet branch (adaptive multi-resolution), fused per scale through channel attention. The cardiac signature is narrowband and non-stationary, buried under a much larger respiratory component — one view is not enough. |
| **C3** | Bidirectional SSM bottleneck | G4 | An 8 s window holds 8–10 cardiac cycles, and a convolutional decoder treats each independently. A bidirectional S4D state-space stack gives a global receptive field in linear time, so beat *n* informs beat *n+1*. **First SSM applied to radar→ECG waveform synthesis.** |
| **C4** | Multi-task decoder + peak-conditioned FiLM | G5, G1 | Three heads — waveform, R-peak heatmap, instantaneous RR. Then the novel part: peak logits feed back through FiLM to modulate the *final* decoder block of the waveform head, so the model is told "a QRS belongs here" **before** it draws one. |
| **C5** | Morphology-aware composite loss | G1 | Huber + multi-resolution STFT + focal BCE on peaks + L1 on RR + peak-weighted L1 + (1 − Pearson). MSE is the conditional mean, so it flattens the R peak — which is exactly why the baseline over-estimates RMSSD by ~2×. |

## ⚠️ Accelerator: **GPU T4 × 2**, Internet **On**, `HF_TOKEN` attached

Please also attach NB02's saved output with **+ Add Input → Notebook Output** before Run All.
The notebook verifies the mounted corpus and only downloads from HF when that input is absent.

## The ablation ladder

This notebook does not just train one model. It trains the **ladder** from `PLAN.md` §7, so the
paper can attribute every point of improvement to a specific component rather than waving at "our
architecture":

1. MultiResLinkNet + MSE — the baseline, our run *(from NB03)*
2. \+ composite loss only *(same backbone, C5)*
3. \+ 8-channel input only *(same backbone, C1)*
4. \+ C1 + C5 together
5. Full CardioMamba-Net, no wavelet branch *(ablates C2)*
6. Full, no SSM bottleneck *(ablates C3)*
7. Full, single-task *(ablates C4)*
8. Full, multi-task but no FiLM *(ablates the refinement specifically)*
9. **Full CardioMamba-Net** *(C1–C5)*
10. Full, Transformer bottleneck instead of SSM — the fair-fight control for C3

Same queue machinery as NB03: completed runs are skipped, the session stops cleanly at the time
budget, and re-running continues. **`QUICK = True` first** — the full model, one fold, 10 epochs,
roughly 30–75 minutes. The earlier smoke cells exercise every ladder variant before this run.

## On `mamba-ssm`

The SSM here is **pure PyTorch** (S4D-Lin as an FFT convolution). It needs no `nvcc`, no custom
CUDA kernel, and it always builds on Kaggle — which the official `mamba-ssm` package frequently
does not. Set `CFG["BOTTLENECK"] = "transformer"` for the attention control. Both are in the ladder.

## Cell-by-cell run guide

| Code cell | What runs | Typical time |
|---:|---|---:|
| 1 | Configuration | < 5 s |
| 2 | Imports, dependency, dual-GPU and disk checks | 1–3 min |
| 3 | Write/import all versioned model/training libraries | 10–30 s |
| 4 | HF login and restore lightweight resume metadata | 1–5 min |
| 5 | Mount attached NB02 corpus or download fallback | < 1 min mounted; 3–12 min fallback |
| 6 | Smoke every architecture; parameters, memory and gradients | 5–15 min |
| 7 | Probe each composite-loss term and head gradient | 2–8 min |
| 8 | Build ablation plus A–F experiment queue | < 10 s |
| 9 | Define split/dataset/recording-safe evaluation helpers | < 10 s |
| 10 | Train/evaluate queue; exact mid-epoch checkpointing | quick: 30–75 min; full: many 10.5 h sessions |
| 11 | Merge NB03 baseline and produce ablation tables | 1–5 min |
| 12 | Draw curves and ablation figures | 2–8 min |
| 13 | Final blocking HF upload and run summary | 2–15 min |

The training cell prints per-epoch ETA. Full mode includes five folds, LOSO subjects, and held-out
scenarios, so it is intentionally resumed across multiple Kaggle sessions.

---
# 1 · Configuration

In [ ]:
CFG = {
    "SRC_REPO":  "Shanmuk4622/cr-rvs-radar-ecg-processed-v2",
    "BASELINE_REPO": "Shanmuk4622/cardiomamba-baselines-v2",
    "DST_REPO":  "Shanmuk4622/cardiomamba-net-v2",
    "HF_PRIVATE": False,
    "RUN_ID":    "nb04_cardiomamba_v2",

    "WORK":    "/kaggle/working/nb04",
    "SCRATCH": "/kaggle/temp/nb04",
    "PUSH_INTERVAL_S": 30 * 60,
    "HF_MAX_UPLOADS_HOUR": 24,

    # ---- architecture (target: < 5 M params, < 1.5 GFLOPs per 8 s window) ----
    "CHANNELS_FULL": ["I", "Q", "phi", "dy", "vel", "acc", "amp", "cardiac"],   # C1
    "CHANNELS_BASE": ["dy"],                                                    # the baseline's input
    # BASE=32 gives ~4.1 M parameters. The 5 M budget is crossed at BASE>=40 (5.57 M),
    # so do not raise this without re-checking the budget cell.
    "BASE":       32,
    "LEVELS":     4,
    "D_SSM":      256,
    "SSM_BLOCKS": 3,
    "D_STATE":    64,
    "DROPOUT":    0.1,

    # ---- composite loss weights (C5). Tune w_stft and w_peakw first. --------
    "W": {"huber": 1.0, "stft": 0.5, "peak": 0.3, "rr": 0.1, "peakw": 0.5, "corr": 0.3},
    "HUBER_DELTA": 0.1,
    "PEAK_WEIGHT": 4.0,

    # ---- training ------------------------------------------------------------
    "EPOCHS":   150,
    "PATIENCE": 25,
    "BATCH":    48,
    "WORKERS":  2,
    "LR":       8e-4,
    "WEIGHT_DECAY": 1e-4,
    "AMP":      True,
    "MULTI_GPU": True,
    "REQUIRE_DUAL_T4": True,
    "SEED":     1337,
    "LOG_EVERY": 25,
    "CHECKPOINT_EVERY_STEPS": 50,
    "CHECKPOINT_EVERY_S": 300,

    # ---- queue ---------------------------------------------------------------
    "EXPERIMENT":  "B_rva",      # the ablation ladder runs on the headline experiment
    "EXTRA_EXPERIMENTS": ["A_resting", "A_valsalva", "A_apnea", "C_all5"],  # full model only
    "RUN_LOSO": True,                 # Experiment D: one full-model run per retained subject
    "RUN_CROSS_SCENARIO": True,       # Experiment F: one held-out scenario per run
    "N_FOLDS":     5,
    "TIME_BUDGET_H": 10.5,
    "QUICK": True,
    "QUICK_EPOCHS": 10,
    "QUICK_FOLDS": 1,
}
import json
print(json.dumps(CFG, indent=2))

In [ ]:
import os, sys, gc, json, math, time, warnings, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone
warnings.filterwarnings("ignore")

def _pip(*p):
    import importlib.util
    miss = [x for x in p if importlib.util.find_spec(x.replace("-", "_")) is None]
    if miss:
        print("installing:", miss)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *miss], check=True)
        for x in miss: __import__(x.replace("-", "_"))
_pip("pyarrow", "huggingface_hub")

import numpy as np, pandas as pd, torch
WORK = Path(CFG["WORK"]); SCRATCH = Path(CFG["SCRATCH"])
for d in (WORK, SCRATCH, WORK / "runs", WORK / "results", WORK / "figures"):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK))

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU{i}: {p.name}  {p.total_memory/2**30:.1f} GB")
    torch.backends.cudnn.benchmark = True
else:
    print("  !! NO GPU -- set Accelerator to 'GPU T4 x2'. Training on CPU is impractical here.")
if torch.cuda.device_count() < 2 or not all("T4" in torch.cuda.get_device_name(i)
                                            for i in range(torch.cuda.device_count())):
    raise RuntimeError("Select Kaggle Accelerator: GPU T4 x2, then restart and Run All.")

---
# 2 · Library

Seven modules. `crvs_cmnet.py` is the new one — it holds the lifting wavelet, the S4D state-space
layer, the FiLM conditioning and the assembled network. The rest are byte-identical to NB03, so
the two notebooks train under exactly the same engine and evaluate with exactly the same metrics.

In [ ]:
MODULES = {
 "crvs_sync.py":    r"""
# crvs_sync.py -- conservative, resumable and interrupt-safe Hugging Face sync.
# A folder upload can involve several HTTP requests, so this deliberately schedules far
# fewer than the nominal API limit: one periodic upload per 30 minutes, plus major stages
# and a best-effort final upload on interrupt. All local writes are atomic.
import os, json, time, random, threading, atexit, signal
from collections import deque
from pathlib import Path
from datetime import datetime, timezone

SYNC_VERSION = 2

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, default=str)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

class RollingLimiter:
    # Limits upload_folder CALLS, not HTTP requests. The low default leaves a wide margin.
    def __init__(self, calls_per_hour=24, min_gap_s=20):
        self.limit = max(1, int(calls_per_hour)); self.min_gap = float(min_gap_s)
        self.times = deque(); self.lock = threading.Lock()
    def take(self, timeout=1800):
        deadline = time.monotonic() + timeout
        while True:
            with self.lock:
                now = time.monotonic()
                while self.times and now - self.times[0] >= 3600:
                    self.times.popleft()
                gap = self.min_gap - (now - self.times[-1]) if self.times else 0.0
                window = 3600 - (now - self.times[0]) if len(self.times) >= self.limit else 0.0
                wait = max(0.0, gap, window)
                if wait <= 0:
                    self.times.append(now); return True
            if time.monotonic() + wait > deadline:
                return False
            time.sleep(min(wait, 10.0))

class HFSync:
    def __init__(self, repo_id, local_dir, token, repo_type="dataset", private=False,
                 run_id="run", push_interval_s=1800, max_upload_calls_hour=24,
                 retry_max=6, verbose=True):
        from huggingface_hub import HfApi
        if not token:
            raise RuntimeError("HF_TOKEN is missing. Add it under Kaggle > Add-ons > Secrets.")
        self.api = HfApi(token=token); self.token = token
        self.repo_id = repo_id; self.repo_type = repo_type; self.private = private
        self.run_id = run_id
        self.local = Path(local_dir); self.local.mkdir(parents=True, exist_ok=True)
        self.interval = max(300, int(push_interval_s))
        self.limiter = RollingLimiter(max_upload_calls_hour)
        self.retry_max = retry_max; self.verbose = verbose
        self._last_push = time.time(); self._dirty = threading.Event()
        self._force = threading.Event(); self._wake = threading.Event()
        self._stop = threading.Event(); self._upload_lock = threading.Lock()
        self._log_lock = threading.Lock(); self._dirty_lock = threading.Lock()
        self._dirty_generation = 0; self._before_final = None
        self._pushes = 0; self._failures = 0; self._closed = False
        self.history = self.local / "sync_history.jsonl"
        self.state_path = self.local / "sync_state.json"
        self._ensure_repo(); self._install_handlers()
        self._thread = threading.Thread(target=self._loop, daemon=True, name="hf-uploader")
        self._thread.start()
        self.log("sync_started", repo=self.repo_id, private=self.private,
                 interval_s=self.interval, sync_version=SYNC_VERSION)

    def _ensure_repo(self):
        from huggingface_hub import create_repo
        create_repo(self.repo_id, repo_type=self.repo_type, private=self.private,
                    exist_ok=True, token=self.token)

    @property
    def url(self):
        kind = "datasets/" if self.repo_type == "dataset" else ""
        return "https://huggingface.co/" + kind + self.repo_id

    def recently_pushed(self, seconds=10):
        return self._pushes > 0 and (time.time() - self._last_push) <= float(seconds)

    def log(self, event, _mark_dirty=True, **kw):
        rec = {"ts": datetime.now(timezone.utc).isoformat(), "run": self.run_id, "event": event}
        rec.update(kw)
        try:
            with self._log_lock, open(self.history, "a", encoding="utf-8") as f:
                f.write(json.dumps(rec, default=str) + "\n"); f.flush()
        except Exception:
            pass
        if _mark_dirty:
            self._touch()
        if self.verbose and event not in ("heartbeat",):
            print("  [" + event + "] " + " ".join(f"{k}={v}" for k, v in kw.items()))

    def _touch(self):
        with self._dirty_lock:
            self._dirty_generation += 1; self._dirty.set()

    def mark_dirty(self, reason=None):
        if reason:
            self.log("dirty", reason=reason)
        else:
            self._touch()

    def save_state(self, state):
        atomic_json(self.state_path, state); self._touch()

    def load_state(self, default=None):
        if self.state_path.exists():
            try:
                return json.loads(self.state_path.read_text(encoding="utf-8"))
            except Exception as e:
                self.log("state_read_error", err=type(e).__name__)
        return default if default is not None else {}

    def pull(self, allow_patterns=None, into=None):
        # Download into the real working folder. Call before producing new local files.
        from huggingface_hub import snapshot_download
        target = Path(into or self.local); target.mkdir(parents=True, exist_ok=True)
        try:
            p = snapshot_download(self.repo_id, repo_type=self.repo_type, token=self.token,
                                  local_dir=str(target), allow_patterns=allow_patterns,
                                  max_workers=4)
            self.log("resume_pull_ok", _mark_dirty=False, path=str(p), patterns=allow_patterns)
            return True
        except Exception as e:
            self.log("resume_pull_empty", _mark_dirty=False,
                     err=f"{type(e).__name__}: {str(e)[:240]}")
            return False

    def set_before_final_flush(self, callback):
        # Trainer registers an atomic emergency-checkpoint callback while it is active.
        self._before_final = callback

    def stage_done(self, name, **kw):
        self.log("stage_done", stage=name, **kw)
        self._force.set(); self._wake.set()

    def _run_final_hook(self):
        cb = self._before_final
        if cb is not None:
            try:
                cb()
            except Exception as e:
                self.log("final_checkpoint_error", err=f"{type(e).__name__}: {e}")

    def _do_upload(self, msg):
        from huggingface_hub import upload_folder
        for attempt in range(self.retry_max):
            if not self.limiter.take(timeout=1800):
                self.log("upload_call_limit_timeout"); return False
            try:
                info = upload_folder(folder_path=str(self.local), repo_id=self.repo_id,
                                     repo_type=self.repo_type, token=self.token,
                                     commit_message=msg,
                                     ignore_patterns=["*.tmp", "**/__pycache__/**", ".git*",
                                                      "*.lock", ".cache/**"])
                self._pushes += 1; self._last_push = time.time()
                meta = {"last_push_utc": datetime.now(timezone.utc).isoformat(),
                        "pushes_this_session": self._pushes,
                        "last_commit": str(getattr(info, "oid", "")), "message": msg}
                atomic_json(self.local / "last_push.json", meta)
                self.log("push_ok", _mark_dirty=False, n=self._pushes,
                         commit=meta["last_commit"], msg=msg)
                return True
            except Exception as e:
                self._failures += 1
                wait = min(300, (2 ** attempt) * 5) * (0.7 + 0.6 * random.random())
                self.log("push_retry", attempt=attempt + 1,
                         err=f"{type(e).__name__}: {str(e)[:500]}", sleep=round(wait, 1))
                time.sleep(wait)
        self.log("push_failed_permanently", msg=msg); return False

    def flush(self, final=False, msg=None, force=False, run_final_hook=False):
        if run_final_hook:
            self._run_final_hook()
        if not self._dirty.is_set() and not force:
            return True
        with self._upload_lock:
            if not self._dirty.is_set() and not force:
                return True
            with self._dirty_lock:
                generation = self._dirty_generation
            stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
            label = "final" if final else "checkpoint"
            message = msg or f"{self.run_id} {label} @ {stamp}Z"
            ok = self._do_upload(message)
            if ok:
                self._force.clear()
                with self._dirty_lock:
                    if self._dirty_generation == generation:
                        self._dirty.clear()
            return ok

    def _loop(self):
        while not self._stop.is_set():
            remaining = max(1.0, self.interval - (time.time() - self._last_push))
            self._wake.wait(min(30.0, remaining)); self._wake.clear()
            if self._stop.is_set():
                break
            forced = self._force.is_set()
            due = (time.time() - self._last_push) >= self.interval
            if self._dirty.is_set() and (due or forced):
                try:
                    tag = "major-stage" if forced else "periodic-30min"
                    self.flush(msg=f"{self.run_id} {tag} @ " +
                               datetime.now(timezone.utc).strftime("%H:%M") + "Z")
                except Exception as e:
                    self.log("loop_error", err=f"{type(e).__name__}: {e}")

    def _install_handlers(self):
        def handler(signum, frame):
            self.log("interrupt", signal=int(signum))
            try:
                self.flush(final=True, force=True, run_final_hook=True,
                           msg=f"{self.run_id} interrupted (signal {signum})")
            finally:
                if signum == signal.SIGINT:
                    raise KeyboardInterrupt
                raise SystemExit(128 + int(signum))
        for sig in (signal.SIGINT, signal.SIGTERM):
            try:
                signal.signal(sig, handler)
            except Exception:
                pass
        atexit.register(self.close)

    def close(self):
        if self._closed:
            return
        self._closed = True; self.log("closing")
        self._stop.set(); self._wake.set()
        try:
            self._thread.join(timeout=5)
            self.flush(final=True, force=self._dirty.is_set(), run_final_hook=True)
        except Exception as e:
            print("Final Hugging Face sync failed:", type(e).__name__, e)
""",
 "crvs_data.py":    r"""
# crvs_data.py -- windowing, folds, normalisation and the torch Dataset.
# Shared by NB03, NB04 and NB05 so every experiment sees byte-identical inputs.
import json, math
import numpy as np
from pathlib import Path

CHANNELS = ["I", "Q", "phi", "dy", "vel", "acc", "amp", "cardiac"]
# One recording is stored as a single UNCOMPRESSED .npy of shape (len(ARRAY_ROWS), n).
# It has to be .npy, not .npz: np.load(..., mmap_mode="r") silently IGNORES mmap_mode on an
# .npz, so every __getitem__ would decompress all 11 arrays to slice 1024 samples out of
# each -- measured at 23 ms per window, which would dominate the GPU time on Kaggle.
ARRAY_ROWS = CHANNELS + ["ecg_norm", "peak_map", "rr_ms"]
ROW = {name: i for i, name in enumerate(ARRAY_ROWS)}
FS       = 128
# Bumped whenever this module changes in a way the notebooks depend on. Every notebook
# asserts it after import, because writing a .py and importing it is NOT idempotent inside
# one kernel: Python caches the module in sys.modules, so a second run silently keeps the
# first version. That is how a stale .npz loader survived a rebuilt notebook once already.
LIB_VERSION = 4
WINDOW   = 1024          # 8.0 s, frozen to Chowdhury et al. 2024 section 2.3.4
HOP_TRAIN = 512          # 50 % overlap on train only
SCENARIOS = ["Resting", "Valsalva", "Apnea", "Tilt-up", "Tilt-down"]

def canon_scenario(s):
    s = str(s).strip().lower()
    for key, out in [("tiltdown", "Tilt-down"), ("tilt_down", "Tilt-down"), ("tilt-down", "Tilt-down"),
                     ("tiltup", "Tilt-up"), ("tilt_up", "Tilt-up"), ("tilt-up", "Tilt-up"),
                     ("valsalva", "Valsalva"), ("apnea", "Apnea"), ("apnoea", "Apnea"),
                     ("rest", "Resting")]:
        if key in s:
            return out
    return str(s)

def range_normalise(x, eps=1e-8):
    # z-score then squash to [-1, 1]; the baseline used [0, 1], we declare the change
    x = np.asarray(x, np.float32)
    sd = float(x.std())
    if not np.isfinite(sd) or sd < eps:
        return np.zeros_like(x, np.float32)          # constant input -> 0, not -1
    x = (x - x.mean()) / (sd + eps)
    lo, hi = np.percentile(x, 0.5), np.percentile(x, 99.5)
    x = np.clip(x, lo, hi)
    rng = float(hi - lo)
    if rng < eps:
        return np.zeros_like(x, np.float32)
    return (2.0 * (x - lo) / rng - 1.0).astype(np.float32)

def peak_heatmap(n, peaks, sigma=3.0):
    # Gaussian bumps at each R peak -- the target for the multi-task peak head
    y = np.zeros(n, np.float32)
    if len(peaks) == 0:
        return y
    half = int(math.ceil(3 * sigma))
    g = np.exp(-0.5 * (np.arange(-half, half + 1) / sigma) ** 2).astype(np.float32)
    for p in np.asarray(peaks, int):
        a, b = max(0, p - half), min(n, p + half + 1)
        y[a:b] = np.maximum(y[a:b], g[a - (p - half): (b - (p - half))])
    return y

def rr_curve(n, peaks, fs=FS, lo_ms=300.0, hi_ms=2000.0):
    # per-sample instantaneous RR interval in ms, linearly interpolated between beats
    out = np.full(n, np.nan, np.float32)
    p = np.asarray(peaks, int)
    if len(p) < 3:
        return np.nan_to_num(out, nan=800.0)
    rr = np.diff(p) / fs * 1000.0
    mid = (p[:-1] + p[1:]) / 2.0
    ok = (rr > lo_ms) & (rr < hi_ms)
    if ok.sum() < 2:
        return np.nan_to_num(out, nan=float(np.median(rr)))
    out = np.interp(np.arange(n), mid[ok], rr[ok]).astype(np.float32)
    return out

_SLOW_WARNED = {"npz": False}

class _Rec:
    # Reads one recording in whichever format is on disk.
    #   .npy (preferred) -- uncompressed, genuinely memory-mapped, ~0.3 ms per window
    #   .npz (legacy)    -- what an earlier NB02 wrote; correct but ~85x slower, because
    #                       np.load ignores mmap_mode on a zip archive and every window
    #                       decompresses all 11 arrays.
    # Both are supported so an existing corpus keeps working without a 400 MB re-upload.
    __slots__ = ("data", "kind")

    def __init__(self, rec_dir, rid):
        d = Path(rec_dir)
        pnpy, pnpz = d / (rid + ".npy"), d / (rid + ".npz")
        if pnpy.exists():
            self.data = np.load(pnpy, mmap_mode="r"); self.kind = "npy"
        elif pnpz.exists():
            self.data = np.load(pnpz); self.kind = "npz"
            if not _SLOW_WARNED["npz"]:
                _SLOW_WARNED["npz"] = True
                print("  note: reading legacy .npz recordings. Correct, but about 85x slower "
                      "per window than .npy -- re-run NB02 to regenerate the corpus and cut "
                      "the data-loading cost.")
        else:
            raise FileNotFoundError(
                f"no recording for '{rid}' in {d} (looked for .npy and .npz). "
                "Either NB02 did not finish, or the snapshot_download allow_patterns in "
                "this notebook do not cover the format NB02 wrote.")

    def rows(self, names, s, e):
        if self.kind == "npy":
            return np.array(self.data[[ROW[n] for n in names], s:e], np.float32)
        return np.stack([np.array(self.data[n][s:e], np.float32) for n in names], 0)

    def one(self, name, s, e):
        if self.kind == "npy":
            return np.array(self.data[ROW[name], s:e], np.float32)
        return np.array(self.data[name][s:e], np.float32)

class WindowDataset:
    # Slices windows on the fly, so changing WINDOW or the overlap never requires
    # re-running NB02.
    def __init__(self, rec_dir, index, norm=None, channels=None, augment=False, seed=0):
        self.rec_dir = Path(rec_dir)
        self.index = index.reset_index(drop=True)
        self.norm = norm
        self.channels = channels or CHANNELS
        self.rows = [ROW[c] for c in self.channels]
        self.augment = augment
        self.seed = int(seed); self.epoch = 0
        self._cache = {}

    def set_epoch(self, epoch):
        # Augmentation is a pure function of (seed, epoch, index). With workers restarted
        # each epoch, an interrupted epoch can replay and skip batches byte-for-byte.
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.index)

    def _rec(self, rid):
        if rid not in self._cache:
            if len(self._cache) > 48:
                self._cache.pop(next(iter(self._cache)))
            self._cache[rid] = _Rec(self.rec_dir, rid)
        return self._cache[rid]

    def __getitem__(self, i):
        import torch
        r = self.index.iloc[i]
        z = self._rec(r["rec_id"])
        s, e = int(r["start"]), int(r["start"]) + WINDOW
        # _Rec.rows / _Rec.one always np.array (copy), never a view into a read-only
        # memmap -- torch.from_numpy on a non-writable array is undefined behaviour.
        x = z.rows(self.channels, s, e)
        if self.norm is not None:
            mu = np.asarray(self.norm["mean"], np.float32)[:, None]
            sd = np.asarray(self.norm["std"], np.float32)[:, None]
            x = (x - mu) / (sd + 1e-6)
        x = np.clip(x, -8.0, 8.0)
        y  = z.one("ecg_norm", s, e)
        pk = z.one("peak_map", s, e)
        rr = z.one("rr_ms", s, e) / 1000.0                           # seconds, O(1) scale
        if self.augment:
            rng = np.random.RandomState(np.random.SeedSequence(
                [self.seed, self.epoch, int(i)]).generate_state(1)[0])
            if rng.rand() < 0.5:
                x = x + rng.randn(*x.shape).astype(np.float32) * 0.01
            if rng.rand() < 0.3:
                g = np.float32(1.0 + 0.1 * rng.randn())
                x = x * g
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(y)[None, :],
                torch.from_numpy(pk)[None, :],
                torch.from_numpy(rr)[None, :])

def compute_norm(rec_dir, index, channels=CHANNELS, max_windows=4000, seed=0):
    # Per-channel mean/std computed on TRAIN WINDOWS ONLY. Computing them over the whole
    # corpus is a classic, invisible source of leakage.
    rng = np.random.RandomState(seed)
    idx = index if len(index) <= max_windows else index.iloc[
        rng.choice(len(index), max_windows, replace=False)]
    n = 0
    s1 = np.zeros(len(channels), np.float64)
    s2 = np.zeros(len(channels), np.float64)
    cache = {}
    rec_dir = Path(rec_dir)
    for _, r in idx.iterrows():
        rid = r["rec_id"]
        if rid not in cache:
            if len(cache) > 48:
                cache.pop(next(iter(cache)))
            cache[rid] = _Rec(rec_dir, rid)
        a, b = int(r["start"]), int(r["start"]) + WINDOW
        x = cache[rid].rows(list(channels), a, b).astype(np.float64)
        s1 += x.sum(1); s2 += (x * x).sum(1); n += x.shape[1]
    mean = s1 / max(n, 1)
    var = np.maximum(s2 / max(n, 1) - mean ** 2, 1e-12)
    return {"mean": mean.tolist(), "std": np.sqrt(var).tolist(),
            "n_samples": int(n), "channels": list(channels)}
""",
 "crvs_metrics.py": r"""
# crvs_metrics.py -- every metric the baseline reports, plus the ones it should have.
import numpy as np
from scipy import signal as ss
from scipy import stats as sstats

def _f(x):
    return np.nan_to_num(np.asarray(x, np.float64), nan=0.0, posinf=0.0, neginf=0.0)

def pearson(a, b):
    a, b = _f(a), _f(b)
    if a.std() < 1e-12 or b.std() < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])

def psd(x, fs=128, nperseg=256):
    f, p = ss.welch(_f(x), fs=fs, nperseg=min(nperseg, len(x)))
    return f, p

def seg_metrics(y, yhat, fs=128):
    # One window. Correlations are reported x100 to match the baseline's tables.
    y, yhat = _f(y), _f(yhat)
    mae = float(np.mean(np.abs(y - yhat)))
    mse = float(np.mean((y - yhat) ** 2))
    cct = 100.0 * pearson(y, yhat)
    _, py = psd(y, fs); _, ph = psd(yhat, fs)
    ccs = 100.0 * pearson(py, ph)
    rms = lambda v: float(np.sqrt(np.mean(np.asarray(v, np.float64) ** 2)))
    rr_t = rms(yhat - y) / (rms(y) + 1e-12)
    rr_s = rms(ph - py) / (rms(py) + 1e-12)
    return {"MAE": mae, "MSE": mse, "CC_temporal": cct, "CC_spectral": ccs,
            "RRMSE_temporal": rr_t, "RRMSE_spectral": rr_s,
            "R2": float(1.0 - np.sum((y - yhat) ** 2) / (np.sum((y - y.mean()) ** 2) + 1e-12))}

def detect_r_peaks(x, fs=128, refractory_s=0.25):
    x = _f(x)
    if len(x) < int(2 * fs):
        return np.array([], int)
    ny = fs / 2.0
    sos = ss.butter(4, [5.0 / ny, min(25.0, ny * 0.95) / ny], btype="band", output="sos")
    b = ss.sosfiltfilt(sos, x)
    e = np.convolve(np.diff(b, prepend=b[0]) ** 2,
                    np.ones(max(1, int(0.10 * fs))) / max(1, int(0.10 * fs)), "same")
    thr = np.percentile(e, 98) * 0.35
    pk, _ = ss.find_peaks(e, height=thr, distance=max(1, int(refractory_s * fs)))
    return pk

def hrv_from_peaks(pk, fs=128):
    out = {"n_peaks": int(len(pk)), "mean_rr_ms": np.nan, "sd_rr_ms": np.nan,
           "mean_hr_bpm": np.nan, "sd_hr_bpm": np.nan, "rmssd_ms": np.nan}
    if len(pk) < 4:
        return out
    rr = np.diff(np.asarray(pk, float)) / fs * 1000.0
    rr = rr[(rr > 300) & (rr < 2000)]
    if len(rr) < 3:
        return out
    hr = 60000.0 / rr
    out.update(mean_rr_ms=float(rr.mean()), sd_rr_ms=float(rr.std()),
               mean_hr_bpm=float(hr.mean()), sd_hr_bpm=float(hr.std()),
               rmssd_ms=float(np.sqrt(np.mean(np.diff(rr) ** 2))))
    return out

def peak_detection_scores(y, yhat, fs=128, tol_ms=100.0):
    # Match predicted R peaks to ground-truth peaks within a tolerance window.
    gt = detect_r_peaks(y, fs); pr = detect_r_peaks(yhat, fs)
    tol = tol_ms / 1000.0 * fs
    used = np.zeros(len(pr), bool)
    tp = 0
    errs = []
    for g in gt:
        if len(pr) == 0:
            break
        # float, not the int64 that find_peaks returns -- assigning np.inf into an
        # integer array raises OverflowError even when the mask selects nothing.
        d = np.abs(pr - g).astype(np.float64)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= tol:
            tp += 1; used[j] = True; errs.append((pr[j] - g) / fs * 1000.0)
    fp = int((~used).sum()); fn = int(len(gt) - tp)
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)
    return {"TP": tp, "FP": fp, "FN": fn, "precision": prec, "recall": rec, "F1": f1,
            "accuracy": tp / max(tp + fp + fn, 1),
            "timing_err_ms_median": float(np.median(np.abs(errs))) if errs else np.nan,
            "timing_err_ms_iqr": float(np.subtract(*np.percentile(np.abs(errs), [75, 25])))
                                  if len(errs) > 3 else np.nan,
            "missed_rate": fn / max(len(gt), 1)}

def aggregate(rows):
    import pandas as pd
    df = pd.DataFrame(rows)
    out = {}
    for c in df.columns:
        if df[c].dtype.kind in "fi":
            out[c] = float(df[c].mean()); out[c + "_std"] = float(df[c].std())
    return out

def bland_altman(a, b):
    a, b = _f(a), _f(b)
    m = (a + b) / 2.0; d = a - b
    bias = float(d.mean()); sd = float(d.std())
    return {"mean": m, "diff": d, "bias": bias, "sd": sd,
            "loa_lo": bias - 1.96 * sd, "loa_hi": bias + 1.96 * sd}

def wilcoxon_holm(groups, better="higher"):
    # Pairwise Wilcoxon signed-rank across folds, Holm-corrected. groups: {name: [values]}
    import itertools
    names = list(groups)
    raw = []
    for a, b in itertools.combinations(names, 2):
        x, y = np.asarray(groups[a], float), np.asarray(groups[b], float)
        n = min(len(x), len(y))
        if n < 3 or np.allclose(x[:n], y[:n]):
            raw.append((a, b, np.nan)); continue
        try:
            p = float(sstats.wilcoxon(x[:n], y[:n]).pvalue)
        except Exception:
            p = np.nan
        raw.append((a, b, p))
    ps = [r[2] for r in raw]
    order = np.argsort([p if np.isfinite(p) else 1.0 for p in ps])
    m = len(ps); adj = [np.nan] * m; run = 0.0
    for k, i in enumerate(order):
        p = ps[i]
        if not np.isfinite(p):
            continue
        run = max(run, (m - k) * p)
        adj[i] = min(1.0, run)
    return [{"a": raw[i][0], "b": raw[i][1], "p": ps[i], "p_holm": adj[i]} for i in range(m)]
""",
 "crvs_models.py":  r"""
# crvs_models.py -- the four baseline 1-D segmentation networks.
# All four are standardised the way Chowdhury et al. 2024 describe (section 3.1):
# 5 levels, 64 filters in the first level, doubling thereafter. Input (B, C_in, 1024).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def cbr(i, o, k=3, s=1):
    return nn.Sequential(nn.Conv1d(i, o, k, s, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))

class DoubleConv(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(cbr(i, o), cbr(o, o))
    def forward(self, x):
        return self.b(x)

# ------------------------------------------------------------------ UNet
class UNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.inc = DoubleConv(in_ch, chs[0])
        self.downs = nn.ModuleList()
        for i in range(levels - 1):
            self.downs.append(DoubleConv(chs[i], chs[i + 1]))
        self.bott = DoubleConv(chs[-1], chs[-1] * 2)
        self.ups = nn.ModuleList()
        self.decs = nn.ModuleList()
        prev = chs[-1] * 2
        for c in reversed(chs):
            self.ups.append(nn.ConvTranspose1d(prev, c, 4, 2, 1))
            self.decs.append(DoubleConv(c * 2, c))
            prev = c
        self.head = nn.Conv1d(chs[0], out_ch, 1)
    def forward(self, x):
        skips = []
        h = self.inc(x); skips.append(h)
        for d in self.downs:
            h = d(F.max_pool1d(h, 2)); skips.append(h)
        h = self.bott(F.max_pool1d(h, 2))
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            h = up(h)
            if h.shape[-1] != sk.shape[-1]:
                h = F.interpolate(h, size=sk.shape[-1], mode="linear", align_corners=False)
            h = dec(torch.cat([h, sk], 1))
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ LinkNet
class LinkEnc(nn.Module):
    def __init__(self, i, o, stride=2):
        super().__init__()
        self.c1 = cbr(i, o, 3, stride)
        self.c2 = nn.Sequential(nn.Conv1d(o, o, 3, 1, 1, bias=False), nn.BatchNorm1d(o))
        self.sc = nn.Sequential(nn.Conv1d(i, o, 1, stride, bias=False), nn.BatchNorm1d(o))
    def forward(self, x):
        return F.relu(self.c2(self.c1(x)) + self.sc(x))

class LinkDec(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        m = max(i // 4, 8)
        self.a = cbr(i, m, 1)
        self.b = nn.Sequential(nn.ConvTranspose1d(m, m, 4, 2, 1, bias=False),
                               nn.BatchNorm1d(m), nn.ReLU(inplace=True))
        self.c = cbr(m, o, 1)
    def forward(self, x):
        return self.c(self.b(self.a(x)))

class LinkNet1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.bott = cbr(prev, prev)
        self.decs = nn.ModuleList()
        rev = list(reversed(chs))
        for k, c in enumerate(rev):
            nxt = rev[k + 1] if k + 1 < len(rev) else chs[0]
            self.decs.append(LinkDec(c, nxt))
        self.head = nn.Sequential(cbr(chs[0], chs[0]), nn.Conv1d(chs[0], out_ch, 1))
    def forward(self, x):
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 2 - k
            if j >= 0:
                s = skips[j]
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                h = h + s
        return {"wave": torch.tanh(self.head(h))}

# ------------------------------------------------------------------ FPN
class FPN1D(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, pyr=128):
        super().__init__()
        chs = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, chs[0], 7, 1)
        self.encs = nn.ModuleList()
        prev = chs[0]
        for c in chs:
            self.encs.append(LinkEnc(prev, c, 2)); prev = c
        self.lat = nn.ModuleList([nn.Conv1d(c, pyr, 1) for c in chs])
        self.smooth = nn.ModuleList([cbr(pyr, pyr) for _ in chs])
        self.heads = nn.ModuleList([nn.Sequential(cbr(pyr, pyr // 2), cbr(pyr // 2, pyr // 2))
                                    for _ in chs])
        self.head = nn.Sequential(cbr(pyr // 2, pyr // 2), nn.Conv1d(pyr // 2, out_ch, 1))
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x); feats = []
        for e in self.encs:
            h = e(h); feats.append(h)
        ps = [None] * len(feats)
        ps[-1] = self.lat[-1](feats[-1])
        for i in range(len(feats) - 2, -1, -1):
            up = F.interpolate(ps[i + 1], size=feats[i].shape[-1], mode="linear",
                               align_corners=False)
            ps[i] = self.lat[i](feats[i]) + up
        ps = [s(p) for s, p in zip(self.smooth, ps)]
        acc = None
        for hd, p in zip(self.heads, ps):
            v = F.interpolate(hd(p), size=L, mode="linear", align_corners=False)
            acc = v if acc is None else acc + v
        return {"wave": torch.tanh(self.head(acc))}

# ------------------------------------------------------------------ MultiResLinkNet
class MultiResBlock(nn.Module):
    # MultiResUNet block (Ibtehaz & Rahman) in 1-D: three successive 3-conv stages of
    # increasing width, concatenated, plus a 1x1 residual shortcut.
    def __init__(self, cin, U, alpha=1.67):
        super().__init__()
        W = alpha * U
        # max(1, ...): below U=4 the 0.167 stage floors to zero channels, and the failure
        # then surfaces as an opaque conv error rather than pointing here.
        c1, c2, c3 = (max(1, int(W * 0.167)), max(1, int(W * 0.333)), max(1, int(W * 0.5)))
        self.out_channels = c1 + c2 + c3
        self.sc = nn.Sequential(nn.Conv1d(cin, self.out_channels, 1, bias=False),
                                nn.BatchNorm1d(self.out_channels))
        self.a = cbr(cin, c1); self.b = cbr(c1, c2); self.c = cbr(c2, c3)
        self.bn1 = nn.BatchNorm1d(self.out_channels)
        self.bn2 = nn.BatchNorm1d(self.out_channels)
    def forward(self, x):
        s = self.sc(x)
        a = self.a(x); b = self.b(a); c = self.c(b)
        o = self.bn1(torch.cat([a, b, c], 1))
        return F.relu(self.bn2(o + s))

class ResPath(nn.Module):
    # Processes an encoder feature before it is added to the decoder, instead of a raw skip.
    def __init__(self, ch, length):
        super().__init__()
        self.blocks = nn.ModuleList()
        for _ in range(max(1, length)):
            self.blocks.append(nn.ModuleDict({
                "sc": nn.Sequential(nn.Conv1d(ch, ch, 1, bias=False), nn.BatchNorm1d(ch)),
                "cv": nn.Sequential(nn.Conv1d(ch, ch, 3, padding=1, bias=False),
                                    nn.BatchNorm1d(ch)),
            }))
    def forward(self, x):
        for b in self.blocks:
            x = F.relu(b["sc"](x) + b["cv"](x))
        return x

class MultiResLinkNet1D(nn.Module):
    # LinkNet skeleton, MultiRes blocks instead of plain convolutions, ResPath skips added
    # (not concatenated), and deep supervision from every encoder level.
    def __init__(self, in_ch=1, out_ch=1, base=64, levels=4, deep_supervision=True):
        super().__init__()
        self.deep_supervision = deep_supervision
        units = [base * (2 ** i) for i in range(levels)]
        self.stem = cbr(in_ch, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        self.bott = MultiResBlock(prev, units[-1])
        self.decs = nn.ModuleList()
        rev_ch = list(reversed(enc_ch))
        cur = self.bott.out_channels
        # Decoder step k must emerge with the channel count AND length of skips[-1-k], or
        # the additive skip is silently dropped and every ResPath receives zero gradient.
        # Encoder here pools AFTER appending the skip, so the target is rev_ch[k] -- not
        # rev_ch[k+1], which is correct only for the stride-2 encoder in LinkNet1D.
        for k in range(levels):
            tgt = rev_ch[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.head = nn.Sequential(cbr(cur, base), nn.Conv1d(base, out_ch, 1))
        self.aux = nn.ModuleList([nn.Conv1d(c, out_ch, 1) for c in enc_ch]) \
                   if deep_supervision else None
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(x)
        skips = []
        for e in self.encs:
            h = e(h); skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.bott(h)
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(                     # raise, not assert: an invariant
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs "        # this
                        f"{s.shape[1]}. Dropping it silently is what cost 59% of this "  # load
                        "model's gradient once already.")   # bearing must survive python -O
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {"wave": torch.tanh(self.head(h))}
        if self.aux is not None and self.training:
            out["aux"] = [F.interpolate(a(s), size=L, mode="linear", align_corners=False)
                          for a, s in zip(self.aux, skips)]
        return out

BASELINES = {"fpn": FPN1D, "unet": UNet1D, "linknet": LinkNet1D,
             "multireslinknet": MultiResLinkNet1D}

def build_baseline(name, in_ch=1, out_ch=1, base=64, levels=4):
    return BASELINES[name](in_ch=in_ch, out_ch=out_ch, base=base, levels=levels)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
""",
 "crvs_cmnet.py":   r"""
# crvs_cmnet.py -- CardioMamba-Net (contributions C1-C5 of PLAN.md).
# C2 dual-domain encoder, C3 bidirectional SSM bottleneck, C4 multi-task decoder with
# peak-conditioned FiLM refinement. C1 lives in the data pipeline, C5 in crvs_losses.
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from crvs_models import cbr, MultiResBlock, ResPath, LinkDec, count_params

class ECA(nn.Module):
    # Efficient channel attention: a length-k 1-D conv over the channel descriptor.
    def __init__(self, ch, k=5):
        super().__init__()
        self.conv = nn.Conv1d(1, 1, k, padding=k // 2, bias=False)
    def forward(self, x):
        w = x.mean(-1, keepdim=True).transpose(1, 2)
        w = torch.sigmoid(self.conv(w)).transpose(1, 2)
        return x * w

class LiftingUnit(nn.Module):
    # Learnable second-generation wavelet: split into even/odd, predict, update.
    # Replaces a fixed wavelet basis with one the network chooses for radar.
    def __init__(self, ch, k=5):
        super().__init__()
        self.P = nn.Sequential(nn.Conv1d(ch, ch, k, padding=k // 2), nn.Tanh(),
                               nn.Conv1d(ch, ch, 1))
        self.U = nn.Sequential(nn.Conv1d(ch, ch, k, padding=k // 2), nn.Tanh(),
                               nn.Conv1d(ch, ch, 1))
    def forward(self, x):
        xe, xo = x[..., ::2], x[..., 1::2]
        n = min(xe.shape[-1], xo.shape[-1])
        xe, xo = xe[..., :n], xo[..., :n]
        d = xo - self.P(xe)
        c = xe + self.U(d)
        return c, d

class WaveletBranch(nn.Module):
    # Multi-resolution analysis producing one feature map per scale, to sit alongside the
    # convolutional branch. LifWavNet uses this idea as the whole network; here it is half
    # of a dual-domain encoder.
    #
    # Level 0 is taken at the INPUT resolution, before any lifting. Encoder level i sits at
    # L/2^i, and a lifting unit halves length, so starting the branch with a lifting step
    # would put every wavelet feature one octave below its conv counterpart and force a 2x
    # upsample at every fusion. The dual-domain claim (C2) is that the two branches see the
    # SAME scale from different domains, so they have to be aligned octave for octave.
    def __init__(self, ch, out_chs, levels=4):
        super().__init__()
        self.proj0 = nn.Conv1d(ch, out_chs[0], 1)
        self.units = nn.ModuleList([LiftingUnit(ch) for _ in range(max(levels - 1, 0))])
        self.proj = nn.ModuleList([nn.Conv1d(ch * 2, o, 1) for o in out_chs[1:]])
    def forward(self, x):
        feats = [self.proj0(x)]
        c = x
        for u, p in zip(self.units, self.proj):
            c, d = u(c)
            feats.append(p(torch.cat([c, d], 1)))
        return feats

class S4D(nn.Module):
    # Diagonal state-space layer (S4D-Lin). Pure PyTorch: an FFT convolution with a kernel
    # built from learned diagonal dynamics. No custom CUDA, so it always builds on Kaggle.
    def __init__(self, d_model, d_state=64, dt_min=1e-3, dt_max=1e-1):
        super().__init__()
        H, N = d_model, d_state // 2
        log_dt = torch.rand(H) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        self.log_dt = nn.Parameter(log_dt)
        self.log_A_real = nn.Parameter(torch.log(0.5 * torch.ones(H, N)))
        self.A_imag = nn.Parameter(math.pi * torch.arange(N).float().repeat(H, 1))
        self.C = nn.Parameter(torch.randn(H, N, 2) * (0.5 ** 0.5))
        self.D = nn.Parameter(torch.randn(H))
    def kernel(self, L, device, dtype=torch.float32):
        dt = torch.exp(self.log_dt).to(dtype).unsqueeze(-1)
        A = -torch.exp(self.log_A_real.to(dtype)) + 1j * self.A_imag.to(dtype)
        C = torch.view_as_complex(self.C.to(dtype).contiguous())
        dtA = A * dt
        n = torch.arange(L, device=device, dtype=dtype)
        K = dtA.unsqueeze(-1) * n
        Cc = C * (torch.exp(dtA) - 1.0) / A
        return 2.0 * torch.einsum("hn,hnl->hl", Cc, torch.exp(K)).real
    def forward(self, u):
        L = u.shape[-1]
        uf = u.float()
        k = self.kernel(L, u.device)
        n = 2 * L
        y = torch.fft.irfft(torch.fft.rfft(uf, n=n) * torch.fft.rfft(k, n=n), n=n)[..., :L]
        y = y + uf * self.D.unsqueeze(-1)
        return y.to(u.dtype)

class BiSSM(nn.Module):
    # Bidirectional SSM block: an 8 s window holds 8-10 cardiac cycles, and this is what
    # lets beat n inform beat n+1. Linear time in sequence length.
    def __init__(self, d, d_state=64, expand=2, dropout=0.1):
        super().__init__()
        self.n1 = nn.LayerNorm(d)
        self.fwd = S4D(d, d_state)
        self.bwd = S4D(d, d_state)
        self.mix = nn.Conv1d(2 * d, d, 1)
        self.n2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Conv1d(d, expand * d, 1), nn.GELU(),
                                nn.Dropout(dropout), nn.Conv1d(expand * d, d, 1))
    def forward(self, x):
        h = self.n1(x.transpose(1, 2)).transpose(1, 2)
        f = self.fwd(h)
        b = self.bwd(h.flip(-1)).flip(-1)
        x = x + self.mix(torch.cat([f, b], 1))
        h = self.n2(x.transpose(1, 2)).transpose(1, 2)
        return x + self.ff(h)

class TransformerBottleneck(nn.Module):
    # The fair-fight control for C3: same budget, attention instead of an SSM.
    def __init__(self, d, nhead=8, layers=3, dropout=0.1, max_len=1024):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, max_len, d) * 0.02)
        lyr = nn.TransformerEncoderLayer(d, nhead, dim_feedforward=2 * d, dropout=dropout,
                                         batch_first=True, norm_first=True,
                                         activation="gelu")
        self.enc = nn.TransformerEncoder(lyr, layers)
    def forward(self, x):
        h = x.transpose(1, 2)
        h = h + self.pos[:, :h.shape[1]]
        return self.enc(h).transpose(1, 2)

class FiLM(nn.Module):
    # Peak-conditioned refinement: the R-peak head tells the waveform head where a QRS
    # belongs BEFORE it draws one. Modulation is per-sample, because a QRS is localised.
    def __init__(self, cond_ch, feat_ch, k=9):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(cond_ch, feat_ch, k, padding=k // 2), nn.GELU(),
                                 nn.Conv1d(feat_ch, 2 * feat_ch, 1))
    def forward(self, feat, cond):
        g, b = self.net(cond).chunk(2, 1)
        return feat * (1.0 + torch.tanh(g)) + b

class CardioMambaNet(nn.Module):
    def __init__(self, in_ch=8, base=32, levels=4, d_ssm=256, ssm_blocks=3, d_state=64,
                 bottleneck="ssm", use_wavelet=True, multitask=True, use_film=True,
                 dropout=0.1):
        super().__init__()
        self.use_wavelet = use_wavelet
        self.multitask = multitask
        self.use_film = use_film and multitask
        units = [base * (2 ** i) for i in range(levels)]
        # C1 lands here: a learnable 1x1 mix over the 8 physics channels, so the network
        # can rediscover arctangent demodulation if that really is optimal.
        self.mix = nn.Sequential(nn.Conv1d(in_ch, 32, 1), nn.GELU())
        self.stem = cbr(32, base, 7, 1)
        self.encs = nn.ModuleList(); self.paths = nn.ModuleList()
        prev = base; enc_ch = []
        for i, u in enumerate(units):
            blk = MultiResBlock(prev, u)
            self.encs.append(blk)
            self.paths.append(ResPath(blk.out_channels, levels - i))
            enc_ch.append(blk.out_channels); prev = blk.out_channels
        if use_wavelet:
            self.wave = WaveletBranch(base, enc_ch, levels)
            self.fuse = nn.ModuleList([nn.Sequential(nn.Conv1d(2 * c, c, 1), ECA(c))
                                       for c in enc_ch])
        self.pre = nn.Conv1d(prev, d_ssm, 1)
        if bottleneck == "ssm":
            self.bott = nn.Sequential(*[BiSSM(d_ssm, d_state, dropout=dropout)
                                        for _ in range(ssm_blocks)])
        elif bottleneck == "transformer":
            self.bott = TransformerBottleneck(d_ssm, layers=ssm_blocks, dropout=dropout)
        else:
            self.bott = nn.Sequential(cbr(d_ssm, d_ssm), cbr(d_ssm, d_ssm))
        self.post = nn.Conv1d(d_ssm, prev, 1)
        self.decs = nn.ModuleList()
        rev = list(reversed(enc_ch)); cur = prev
        # Same indexing rule as MultiResLinkNet1D: target rev[k], so decoder step k lines up
        # with skips[-1-k] in both channels and length and the ResPath actually contributes.
        for k in range(levels):
            tgt = rev[k]
            self.decs.append(LinkDec(cur, tgt)); cur = tgt
        self.refine = cbr(cur, base)
        self.head_wave = nn.Conv1d(base, 1, 1)
        if multitask:
            self.head_peak = nn.Sequential(cbr(cur, base), nn.Conv1d(base, 1, 1))
            rr_ch = max(base // 2, 4)
            self.head_rr = nn.Sequential(cbr(cur, rr_ch), nn.Conv1d(rr_ch, 1, 1))
            if self.use_film:
                self.film = FiLM(1, base)
    def forward(self, x):
        L = x.shape[-1]
        h = self.stem(self.mix(x))
        wfeat = self.wave(h) if self.use_wavelet else None
        skips = []
        for i, e in enumerate(self.encs):
            h = e(h)
            if wfeat is not None:
                w = wfeat[i]
                if w.shape[-1] != h.shape[-1]:
                    w = F.interpolate(w, size=h.shape[-1], mode="linear", align_corners=False)
                h = self.fuse[i](torch.cat([h, w], 1))
            skips.append(h)
            h = F.max_pool1d(h, 2)
        h = self.post(self.bott(self.pre(h)))
        for k, d in enumerate(self.decs):
            h = d(h)
            j = len(skips) - 1 - k
            if j >= 0:
                s = self.paths[j](skips[j])
                if h.shape[-1] != s.shape[-1]:
                    h = F.interpolate(h, size=s.shape[-1], mode="linear", align_corners=False)
                if h.shape[1] != s.shape[1]:
                    raise RuntimeError(
                        f"skip channel mismatch at decoder {k}: {h.shape[1]} vs {s.shape[1]}")
                h = h + s
        if h.shape[-1] != L:
            h = F.interpolate(h, size=L, mode="linear", align_corners=False)
        out = {}
        if self.multitask:
            peak_logit = self.head_peak(h)
            out["peak"] = peak_logit
            out["rr"] = F.softplus(self.head_rr(h))
            f = self.refine(h)
            if self.use_film:
                # Gradient flows through the conditioning on purpose: the peak head is meant
                # to be shaped by the waveform loss as well as its own, which is the point of
                # peak-conditioned refinement. Detaching here would make it a one-way hint.
                f = self.film(f, torch.sigmoid(peak_logit))
            out["wave"] = torch.tanh(self.head_wave(f))
        else:
            out["wave"] = torch.tanh(self.head_wave(self.refine(h)))
        return out

def build_cmnet(**kw):
    return CardioMambaNet(**kw)
""",
 "crvs_losses.py":  r"""
# crvs_losses.py -- C5, the morphology-aware composite loss.
# Plain MSE is the conditional mean, so it flattens the R peak; that is exactly why the
# baseline over-estimates RMSSD by ~2x. Every term here exists to stop that.
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiResSTFTLoss(nn.Module):
    # Spectral convergence + log-magnitude at three resolutions. Forces the model to get
    # the spectrum right, not just the sample-wise average.
    def __init__(self, ffts=(256, 128, 64)):
        super().__init__()
        self.ffts = ffts
    def _one(self, y, yh, n):
        hop, win = n // 4, n
        w = torch.hann_window(win, device=y.device, dtype=torch.float32)
        kw = dict(n_fft=n, hop_length=hop, win_length=win, window=w,
                  return_complex=True, center=True, pad_mode="reflect")
        Y = torch.stft(y, **kw).abs().clamp_min(1e-7)
        H = torch.stft(yh, **kw).abs().clamp_min(1e-7)
        sc = torch.norm(Y - H, p="fro", dim=(-2, -1)) / (torch.norm(Y, p="fro", dim=(-2, -1)) + 1e-7)
        mag = F.l1_loss(torch.log(H), torch.log(Y))
        return sc.mean() + mag
    def forward(self, y, yh):
        y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
        return sum(self._one(y, yh, n) for n in self.ffts) / len(self.ffts)

def pearson_loss(y, yh, eps=1e-8):
    y = y.squeeze(1).float(); yh = yh.squeeze(1).float()
    y = y - y.mean(-1, keepdim=True); yh = yh - yh.mean(-1, keepdim=True)
    num = (y * yh).sum(-1)
    den = y.norm(dim=-1) * yh.norm(dim=-1) + eps
    return (1.0 - num / den).mean()

def focal_bce(logit, target, alpha=0.75, gamma=2.0):
    p = torch.sigmoid(logit)
    ce = F.binary_cross_entropy_with_logits(logit, target, reduction="none")
    pt = p * target + (1 - p) * (1 - target)
    w = alpha * target + (1 - alpha) * (1 - target)
    return (w * (1 - pt).pow(gamma) * ce).mean()

class CompositeLoss(nn.Module):
    def __init__(self, w_huber=1.0, w_stft=0.5, w_peak=0.3, w_rr=0.1,
                 w_peakw=0.5, w_corr=0.3, huber_delta=0.1, peak_weight=4.0):
        super().__init__()
        self.w = dict(huber=w_huber, stft=w_stft, peak=w_peak, rr=w_rr,
                      peakw=w_peakw, corr=w_corr)
        self.delta = huber_delta
        self.peak_weight = peak_weight
        self.stft = MultiResSTFTLoss()
    def forward(self, pred, y, pk=None, rr=None):
        parts = {}
        wave = pred["wave"]
        if self.w["huber"]:
            parts["huber"] = F.huber_loss(wave, y, delta=self.delta)
        if self.w["stft"]:
            parts["stft"] = self.stft(y, wave)
        if self.w["corr"]:
            parts["corr"] = pearson_loss(y, wave)
        if self.w["peakw"] and pk is not None:
            wgt = 1.0 + self.peak_weight * pk
            parts["peakw"] = ((wgt * (wave - y).abs()).sum() / (wgt.sum() + 1e-8))
        if self.w["peak"] and pk is not None and "peak" in pred:
            parts["peak"] = focal_bce(pred["peak"], pk)
        if self.w["rr"] and rr is not None and "rr" in pred:
            parts["rr"] = F.l1_loss(pred["rr"], rr)
        if "aux" in pred:
            parts["aux"] = sum(F.huber_loss(a, y, delta=self.delta)
                               for a in pred["aux"]) / max(len(pred["aux"]), 1) * 0.2
        total = sum(self.w.get(k, 1.0) * v for k, v in parts.items())
        return total, {k: float(v.detach()) for k, v in parts.items()}

class MSEOnly(nn.Module):
    # The baseline's objective, kept verbatim so ablation row 1 is a true reproduction.
    def forward(self, pred, y, pk=None, rr=None):
        l = F.mse_loss(pred["wave"], y)
        if "aux" in pred:
            l = l + 0.2 * sum(F.mse_loss(a, y) for a in pred["aux"]) / max(len(pred["aux"]), 1)
        return l, {"mse": float(l.detach())}
""",
 "crvs_engine.py":  r"""
# crvs_engine.py -- deterministic dual-GPU engine with mid-epoch recovery and telemetry.
import csv, json, math, time, os, random, shutil, subprocess, threading, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ENGINE_VERSION = 4

def _autocast(device_type, enabled):
    try:
        return torch.amp.autocast(device_type=device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast(enabled=enabled)

def _grad_scaler(device_type, enabled):
    try:
        return torch.amp.GradScaler(device_type, enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

def pick_device():
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        names = [torch.cuda.get_device_name(i) for i in range(n)]
        return torch.device("cuda"), n, names
    return torch.device("cpu"), 0, []

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def _atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(value, f, indent=2, default=str, allow_nan=True)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def _jsonable(v):
    if isinstance(v, (np.floating, np.integer)): return v.item()
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, float) and not math.isfinite(v): return None
    return v

def _append_jsonl(path, record):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    clean = {str(k): _jsonable(v) for k, v in record.items()}
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(clean, default=str, allow_nan=False) + "\n"); f.flush()

def _mean_parts(sums, n, prefix):
    return {f"{prefix}_{k}": float(v) / max(int(n), 1) for k, v in sums.items()}

def _system_stats(out_dir):
    d = shutil.disk_usage(Path(out_dir))
    rec = {"disk_free_gb": d.free / 2**30, "disk_used_gb": d.used / 2**30}
    try:
        import resource
        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        rec["process_peak_rss_gb"] = rss / 2**20  # Linux ru_maxrss is KiB
    except Exception:
        pass
    try:
        load = os.getloadavg(); rec.update(cpu_load_1m=load[0], cpu_load_5m=load[1], cpu_load_15m=load[2])
    except Exception:
        pass
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            rec[f"gpu{i}_peak_alloc_gb"] = torch.cuda.max_memory_allocated(i) / 2**30
            rec[f"gpu{i}_peak_reserved_gb"] = torch.cuda.max_memory_reserved(i) / 2**30
        try:
            q = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,temperature.gpu,power.draw,memory.used",
                                "--format=csv,noheader,nounits"], capture_output=True, text=True,
                               timeout=10, check=False)
            for line in q.stdout.strip().splitlines():
                vals = [x.strip() for x in line.split(",")]
                if len(vals) == 5:
                    i = vals[0]
                    for key, val in zip(("util_pct", "temp_c", "power_w", "mem_used_mb"), vals[1:]):
                        try: rec[f"gpu{i}_{key}"] = float(val)
                        except ValueError: pass
        except Exception:
            pass
    return rec

class Trainer:
    def __init__(self, model, loss_fn, out_dir, run_id, sync=None, lr=5e-4, weight_decay=1e-4,
                 epochs=120, patience=20, batch_size=64, num_workers=2, amp=True,
                 multi_gpu=True, grad_clip=1.0, min_lr=1e-6, log_every=25,
                 checkpoint_every_steps=50, checkpoint_every_s=300, seed=42,
                 require_dual_gpu=False, run_config=None):
        self.device, self.ngpu, self.gpu_names = pick_device()
        if require_dual_gpu and self.ngpu < 2:
            raise RuntimeError("This training notebook requires Kaggle GPU T4 x2. "
                               "Choose Settings > Accelerator > GPU T4 x2, then restart.")
        self.raw_model = model.to(self.device); self.model = self.raw_model
        if multi_gpu and self.ngpu > 1:
            self.model = nn.DataParallel(self.raw_model)
        self.loss_fn = loss_fn
        self.out = Path(out_dir); self.out.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id; self.sync = sync
        self.epochs = int(epochs); self.patience = int(patience)
        self.bs = int(batch_size); self.nw = int(num_workers); self.seed = int(seed)
        self.amp = bool(amp and self.device.type == "cuda"); self.grad_clip = grad_clip
        self.opt = torch.optim.AdamW(self.raw_model.parameters(), lr=lr, weight_decay=weight_decay)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.opt, T_max=self.epochs, eta_min=min_lr)
        self.scaler = _grad_scaler(self.device.type, self.amp)
        self.log_every = max(1, int(log_every))
        self.checkpoint_every_steps = max(1, int(checkpoint_every_steps))
        self.checkpoint_every_s = max(30, int(checkpoint_every_s))
        self.config = dict(run_config or {})
        self.config_hash = hashlib.sha256(json.dumps(
            self.config, sort_keys=True, default=str).encode()).hexdigest()
        self.state = {"schema_version": 4, "engine_version": ENGINE_VERSION,
                      "epoch": 0, "active_epoch": 0, "batch_in_epoch": 0,
                      "global_step": 0, "best": float("inf"), "best_epoch": -1,
                      "bad_epochs": 0, "history": [], "partial": {},
                      "run_id": run_id, "done": False, "config_hash": self.config_hash,
                      "created_utc": datetime.now(timezone.utc).isoformat()}
        self._save_lock = threading.Lock(); self._last_checkpoint = time.time()
        self._active = False
        _atomic_json(self.out / "run_config.json", self.config)
        _atomic_json(self.out / "environment.json", {
            "engine_version": ENGINE_VERSION, "torch": torch.__version__,
            "cuda": torch.version.cuda, "gpu_count": self.ngpu, "gpu_names": self.gpu_names,
            "amp": self.amp, "python": os.sys.version, "config_hash": self.config_hash})

    @property
    def ckpt(self):
        return self.out / "state.pt"

    def _payload(self):
        return {"model": self.raw_model.state_dict(), "opt": self.opt.state_dict(),
                "sched": self.sched.state_dict(), "scaler": self.scaler.state_dict(),
                "state": self.state, "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
                "np_rng": np.random.get_state(), "python_rng": random.getstate(),
                "config": self.config, "config_hash": self.config_hash,
                "saved_utc": datetime.now(timezone.utc).isoformat()}

    def save(self, tag="state", reason="checkpoint"):
        with self._save_lock:
            path = self.out / (tag + ".pt"); tmp = path.with_suffix(".pt.tmp")
            torch.save(self._payload(), tmp); os.replace(tmp, path)
            _atomic_json(self.out / "state.json", self.state)
            self._last_checkpoint = time.time()
        if self.sync:
            self.sync.mark_dirty(f"{self.run_id}:{reason}")

    def emergency_checkpoint(self):
        if self._active:
            self.save("state", reason="interrupt-emergency")

    def load(self):
        if not self.ckpt.exists():
            return False
        try:
            d = torch.load(self.ckpt, map_location=self.device, weights_only=False)
            got_hash = d.get("config_hash", d.get("state", {}).get("config_hash"))
            if got_hash and got_hash != self.config_hash:
                raise RuntimeError("checkpoint configuration differs from this run. "
                                   "Use a new RUN_ID or restore the original configuration.")
            self.raw_model.load_state_dict(d["model"], strict=True)
            self.opt.load_state_dict(d["opt"]); self.sched.load_state_dict(d["sched"])
            self.scaler.load_state_dict(d["scaler"]); self.state = d["state"]
            torch.set_rng_state(d["torch_rng"].cpu()); np.random.set_state(d["np_rng"])
            random.setstate(d["python_rng"])
            if torch.cuda.is_available() and d.get("cuda_rng"):
                torch.cuda.set_rng_state_all([x.cpu() for x in d["cuda_rng"]])
            print(f"  resumed {self.run_id}: completed_epoch={self.state['epoch']}, "
                  f"active_epoch={self.state.get('active_epoch')}, "
                  f"completed_batches={self.state.get('batch_in_epoch', 0)}")
            return True
        except Exception as e:
            raise RuntimeError(f"Checkpoint exists but cannot be resumed safely: "
                               f"{type(e).__name__}: {e}") from e

    def _loader(self, ds, shuffle, epoch=0):
        if len(ds) == 0:
            raise RuntimeError("empty dataset -- check the subject split")
        if hasattr(ds, "set_epoch"):
            ds.set_epoch(epoch)
        drop = bool(shuffle) and len(ds) > self.bs
        gen = torch.Generator(); gen.manual_seed(self.seed + int(epoch) * 1000003)
        return DataLoader(ds, batch_size=min(self.bs, max(len(ds), 1)), shuffle=shuffle,
                          generator=gen, num_workers=self.nw,
                          pin_memory=(self.device.type == "cuda"), drop_last=drop,
                          persistent_workers=False)

    def _step(self, batch, train):
        x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
        with _autocast(self.device.type, self.amp):
            pred = self.model(x)
            if isinstance(pred, dict) and "aux" in pred and not train:
                pred = {k: v for k, v in pred.items() if k != "aux"}
            loss, parts = self.loss_fn(pred, y, pk, rr)
        return loss, {k: float(v) for k, v in parts.items()}, pred, y, pk, rr

    def _write_batch(self, rec):
        _append_jsonl(self.out / "batch_metrics.jsonl", rec)

    def _validation(self, val_ds, epoch):
        self.model.eval(); sums = {}; n = 0; Y = []; P = []; PK = []; PH = []; RR = []; RH = []
        t0 = time.time()
        with torch.no_grad():
            for batch in self._loader(val_ds, False, epoch):
                loss, parts, pred, y, pk, rr = self._step(batch, False)
                parts = {"total": float(loss), **parts}
                for k, v in parts.items(): sums[k] = sums.get(k, 0.0) + float(v)
                n += 1; Y.append(y.squeeze(1).float().cpu().numpy())
                P.append(pred["wave"].squeeze(1).float().cpu().numpy())
                PK.append(pk.squeeze(1).float().cpu().numpy())
                if "peak" in pred: PH.append(pred["peak"].squeeze(1).float().cpu().numpy())
                RR.append(rr.squeeze(1).float().cpu().numpy())
                if "rr" in pred: RH.append(pred["rr"].squeeze(1).float().cpu().numpy())
        if n == 0: raise RuntimeError("validation loader yielded zero batches")
        y = np.concatenate(Y); p = np.concatenate(P); pk = np.concatenate(PK)
        rec = _mean_parts(sums, n, "val")
        rec["val_seconds"] = time.time() - t0; rec["val_windows"] = len(y)
        rec.update(val_true_mean=float(y.mean()), val_true_std=float(y.std()),
                   val_true_min=float(y.min()), val_true_max=float(y.max()),
                   val_pred_mean=float(p.mean()), val_pred_std=float(p.std()),
                   val_pred_min=float(p.min()), val_pred_max=float(p.max()),
                   val_pred_bias=float((p-y).mean()))
        try:
            from crvs_metrics import seg_metrics, peak_detection_scores, detect_r_peaks, hrv_from_peaks
            global_m = seg_metrics(y.reshape(-1), p.reshape(-1))
            rec.update({"val_" + k: v for k, v in global_m.items()})
            # Per-window waveform diagnostics are compact and retained for every epoch.
            den_y = np.sqrt(np.sum((y - y.mean(1, keepdims=True)) ** 2, axis=1))
            den_p = np.sqrt(np.sum((p - p.mean(1, keepdims=True)) ** 2, axis=1))
            cc = np.sum((y-y.mean(1, keepdims=True))*(p-p.mean(1, keepdims=True)), axis=1) / (den_y*den_p+1e-12)
            win = {"epoch": np.full(len(y), epoch + 1), "window": np.arange(len(y)),
                   "mae": np.mean(np.abs(y-p), 1), "mse": np.mean((y-p)**2, 1),
                   "cc_temporal": 100*cc}
            try:
                import pandas as pd
                frame = pd.DataFrame(win)
                if hasattr(val_ds, "index") and len(val_ds.index) == len(frame):
                    for col in ("rec_id", "subject", "scenario_canon", "start"):
                        if col in val_ds.index: frame[col] = val_ds.index[col].to_numpy()
                vd = self.out / "validation_windows"; vd.mkdir(exist_ok=True)
                frame.to_parquet(vd / f"epoch_{epoch+1:04d}.parquet", index=False)
            except Exception as e:
                rec["val_window_table_error"] = f"{type(e).__name__}: {e}"
            # Never concatenate different recordings: that fabricates a beat interval at
            # each boundary. Compute peak/HRV metrics per recording, then macro-average.
            record_rows = []
            if hasattr(val_ds, "index") and len(val_ds.index) == len(y):
                ix = val_ds.index.reset_index(drop=True).assign(_row=np.arange(len(y)))
                for rid, grp in ix.groupby("rec_id"):
                    pos = grp.sort_values("start")["_row"].to_numpy()
                    if len(pos) < 2: continue
                    yg = np.concatenate(y[pos]); pg = np.concatenate(p[pos])
                    peak_s = peak_detection_scores(yg, pg)
                    gt_hrv = hrv_from_peaks(detect_r_peaks(yg))
                    pr_hrv = hrv_from_peaks(detect_r_peaks(pg))
                    rr = {"rec_id": rid, "subject": str(grp["subject"].iloc[0]), **peak_s}
                    for k in ("mean_rr_ms", "sd_rr_ms", "mean_hr_bpm", "sd_hr_bpm", "rmssd_ms"):
                        rr[f"true_{k}"] = gt_hrv[k]; rr[f"pred_{k}"] = pr_hrv[k]
                        rr[f"abs_error_{k}"] = abs(pr_hrv[k]-gt_hrv[k])
                    record_rows.append(rr)
            if record_rows:
                import pandas as pd
                rdf = pd.DataFrame(record_rows)
                rd = self.out / "validation_recordings"; rd.mkdir(exist_ok=True)
                rdf.to_parquet(rd / f"epoch_{epoch+1:04d}.parquet", index=False)
                for k in ("TP", "FP", "FN", "precision", "recall", "F1", "accuracy",
                          "timing_err_ms_median", "timing_err_ms_iqr", "missed_rate"):
                    rec[f"val_wave_peak_{k}"] = float(rdf[k].mean())
                for k in ("mean_rr_ms", "sd_rr_ms", "mean_hr_bpm", "sd_hr_bpm", "rmssd_ms"):
                    rec[f"val_hrv_true_{k}"] = float(rdf[f"true_{k}"].mean())
                    rec[f"val_hrv_pred_{k}"] = float(rdf[f"pred_{k}"].mean())
                    rec[f"val_hrv_abs_error_{k}"] = float(rdf[f"abs_error_{k}"].mean())
        except Exception as e:
            rec["val_signal_metrics_error"] = f"{type(e).__name__}: {e}"
        if PH:
            ph = np.concatenate(PH); prob = 1 / (1 + np.exp(-np.clip(ph, -30, 30)))
            truth = pk >= 0.5; guess = prob >= 0.5
            tp = int(np.sum(truth & guess)); fp = int(np.sum(~truth & guess)); fn = int(np.sum(truth & ~guess))
            prec = tp / max(tp+fp, 1); recall = tp / max(tp+fn, 1)
            rec.update(val_peak_head_TP=tp, val_peak_head_FP=fp, val_peak_head_FN=fn,
                       val_peak_head_precision=prec, val_peak_head_recall=recall,
                       val_peak_head_F1=2*prec*recall/max(prec+recall, 1e-12))
        if RH:
            rr = np.concatenate(RR); rh = np.concatenate(RH); mask = rr > 0
            if np.any(mask):
                rec["val_rr_head_mae_ms"] = float(np.mean(np.abs(rr[mask]-rh[mask]))*1000)
                rec["val_rr_head_rmse_ms"] = float(np.sqrt(np.mean((rr[mask]-rh[mask])**2))*1000)
        return rec

    def _write_epoch(self, rec):
        _append_jsonl(self.out / "epoch_metrics.jsonl", rec)
        # CSV is convenient in Kaggle; JSONL remains the lossless schema-of-record.
        rows = self.state["history"]
        keys = sorted({k for r in rows for k in r})
        tmp = self.out / "epoch_metrics.csv.tmp"
        with open(tmp, "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys); w.writeheader()
            for row in rows: w.writerow({k: _jsonable(row.get(k)) for k in keys})
            f.flush(); os.fsync(f.fileno())
        os.replace(tmp, self.out / "epoch_metrics.csv")

    def fit(self, train_ds, val_ds):
        start = int(self.state["epoch"])
        if start >= self.epochs:
            print(f"  {self.run_id} already complete at epoch {start}/{self.epochs}")
            return self.state
        if self.state.get("done"):
            self.state["done"] = False
        self._active = True
        if self.sync: self.sync.set_before_final_flush(self.emergency_checkpoint)
        run_t0 = time.time()
        try:
            for ep in range(start, self.epochs):
                if torch.cuda.is_available():
                    for gpu_i in range(torch.cuda.device_count()):
                        torch.cuda.reset_peak_memory_stats(gpu_i)
                tl = self._loader(train_ds, True, ep)
                resume_batch = int(self.state.get("batch_in_epoch", 0)) if int(self.state.get("active_epoch", ep)) == ep else 0
                partial = self.state.get("partial", {}) if resume_batch else {}
                sums = {k: float(v) for k, v in partial.get("sums", {}).items()}
                n = int(partial.get("n", 0)); samples = int(partial.get("samples", 0))
                grad_sum = float(partial.get("grad_sum", 0)); clip_events = int(partial.get("clip_events", 0))
                epoch_t0 = time.time(); data_t = 0.0; compute_t = 0.0; last_end = time.time()
                self.state.update(active_epoch=ep, batch_in_epoch=resume_batch, done=False)
                self.model.train()
                for i, batch in enumerate(tl):
                    data_t += time.time() - last_end
                    if i < resume_batch:
                        last_end = time.time(); continue
                    step_t0 = time.time(); self.opt.zero_grad(set_to_none=True)
                    loss, parts, _, _, _, _ = self._step(batch, True)
                    if not torch.isfinite(loss):
                        self.save("state", reason="non-finite-loss")
                        raise FloatingPointError(f"non-finite loss at epoch {ep+1}, batch {i+1}")
                    self.scaler.scale(loss).backward(); self.scaler.unscale_(self.opt)
                    grad = float(torch.nn.utils.clip_grad_norm_(
                        self.raw_model.parameters(), self.grad_clip or float("inf")))
                    if self.grad_clip and grad > self.grad_clip: clip_events += 1
                    self.scaler.step(self.opt); self.scaler.update()
                    compute_t += time.time() - step_t0
                    values = {"total": float(loss.detach()), **parts}
                    for k, v in values.items(): sums[k] = sums.get(k, 0.0) + float(v)
                    n += 1; samples += int(batch[0].shape[0]); grad_sum += grad
                    self.state["global_step"] = int(self.state.get("global_step", 0)) + 1
                    self.state["batch_in_epoch"] = i + 1
                    self.state["partial"] = {"sums": sums, "n": n, "samples": samples,
                                             "grad_sum": grad_sum, "clip_events": clip_events}
                    if (i + 1) % self.log_every == 0 or i + 1 == len(tl):
                        brec = {"ts": datetime.now(timezone.utc).isoformat(), "run_id": self.run_id,
                                "epoch": ep+1, "batch": i+1, "batches": len(tl),
                                "global_step": self.state["global_step"], "loss": float(loss),
                                "grad_norm": grad, "lr": self.opt.param_groups[0]["lr"],
                                "amp_scale": float(self.scaler.get_scale()),
                                "windows_per_s": int(batch[0].shape[0])/max(time.time()-step_t0, 1e-9)}
                        brec.update({"loss_"+k: v for k, v in parts.items()}); self._write_batch(brec)
                    due_step = self.state["global_step"] % self.checkpoint_every_steps == 0
                    due_time = time.time() - self._last_checkpoint >= self.checkpoint_every_s
                    if due_step or due_time:
                        self.save("state", reason="mid-epoch")
                    last_end = time.time()
                if n == 0: raise RuntimeError("training loader yielded zero batches")
                self.sched.step(); val = self._validation(val_ds, ep)
                rec = {"ts": datetime.now(timezone.utc).isoformat(), "run_id": self.run_id,
                       "epoch": ep+1, "epochs_planned": self.epochs,
                       "global_step": self.state["global_step"], "train_batches": n,
                       "train_windows": samples, "train_grad_norm_mean": grad_sum/max(n,1),
                       "train_grad_clip_events": clip_events,
                       "train_data_seconds": data_t, "train_compute_seconds": compute_t,
                       "train_windows_per_s": samples/max(compute_t, 1e-9),
                       "epoch_seconds": time.time()-epoch_t0,
                       "elapsed_seconds": time.time()-run_t0,
                       "lr": self.opt.param_groups[0]["lr"],
                       "amp_scale": float(self.scaler.get_scale())}
                rec.update(_mean_parts(sums, n, "train")); rec.update(val); rec.update(_system_stats(self.out))
                with torch.no_grad():
                    rec["model_parameter_l2"] = math.sqrt(sum(
                        float(torch.sum(p.detach().float() ** 2)) for p in self.raw_model.parameters()))
                va = float(rec["val_total"])
                improved = va < float(self.state["best"]) - 1e-6
                if improved:
                    self.state["best"] = va; self.state["best_epoch"] = ep+1
                    self.state["bad_epochs"] = 0
                else:
                    self.state["bad_epochs"] = int(self.state.get("bad_epochs", 0)) + 1
                rec.update(improved=bool(improved), best_val=float(self.state["best"]),
                           best_epoch=int(self.state["best_epoch"]),
                           bad_epochs=int(self.state["bad_epochs"]))
                self.state["epoch"] = ep+1; self.state["active_epoch"] = ep+1
                self.state["batch_in_epoch"] = 0; self.state["partial"] = {}
                self.state["history"].append(rec)
                if improved: self.save("best", reason="new-best-local")
                self.save("state", reason="epoch-complete"); self._write_epoch(rec)
                if self.sync:
                    self.sync.log("epoch", run_id=self.run_id, epoch=ep+1,
                                  train_total=rec.get("train_total"), val_total=va,
                                  cc_t=rec.get("val_CC_temporal"), cc_s=rec.get("val_CC_spectral"),
                                  hr_mae_bpm=rec.get("val_hrv_abs_error_mean_hr_bpm"), improved=improved)
                eta = rec["epoch_seconds"] * max(self.epochs-ep-1, 0) / 3600
                print(f"  ep {ep+1:>3}/{self.epochs} train {rec['train_total']:.5f} "
                      f"val {va:.5f} CCt {rec.get('val_CC_temporal', float('nan')):.1f} "
                      f"CCs {rec.get('val_CC_spectral', float('nan')):.1f} "
                      f"{'*' if improved else ''} {rec['epoch_seconds']:.0f}s ETA {eta:.1f}h")
                if int(self.state["bad_epochs"]) >= self.patience:
                    self.state["stop_reason"] = "early_stopping"; break
            self.state["done"] = True
            self.state["finished_utc"] = datetime.now(timezone.utc).isoformat()
            self.save("state", reason="run-complete")
            if self.sync: self.sync.log("training_complete", run_id=self.run_id)
            return self.state
        except KeyboardInterrupt:
            # SIGINT normally reaches HFSync first: its final hook already saved and pushed.
            # Avoid a duplicate commit; a directly-raised KeyboardInterrupt still takes this path.
            if time.time() - self._last_checkpoint > 2:
                self.save("state", reason="keyboard-interrupt")
            if self.sync:
                if not self.sync.recently_pushed(5):
                    self.sync.flush(final=True, force=True,
                                    msg=f"{self.run_id} stopped by user", run_final_hook=False)
            raise
        except Exception as e:
            self.state["last_error"] = f"{type(e).__name__}: {e}"
            self.save("state", reason="training-error")
            if self.sync:
                self.sync.log("training_error", run_id=self.run_id, error=self.state["last_error"])
                self.sync.flush(final=True, force=True,
                                msg=f"{self.run_id} error checkpoint", run_final_hook=False)
            raise
        finally:
            self._active = False
            if self.sync: self.sync.set_before_final_flush(None)

    @torch.no_grad()
    def predict(self, ds, max_keep=None):
        bp = self.out / "best.pt"
        if bp.exists():
            d = torch.load(bp, map_location=self.device, weights_only=False)
            self.raw_model.load_state_dict(d["model"], strict=True)
        self.model.eval(); Y = []; P = []
        for batch in self._loader(ds, False, 0):
            x, y, pk, rr = [b.to(self.device, non_blocking=True) for b in batch]
            with _autocast(self.device.type, self.amp): out = self.model(x)
            Y.append(y.squeeze(1).float().cpu().numpy())
            P.append(out["wave"].squeeze(1).float().cpu().numpy())
        Y = np.concatenate(Y); P = np.concatenate(P)
        if max_keep is not None: return Y[:max_keep], P[:max_keep]
        return Y, P
""",
}
for nm, src in MODULES.items():
    (WORK / nm).write_text(src)
    print(f"  {nm:<20} {len(src):>7,} chars")
import hashlib
MODULE_HASHES = {nm: hashlib.sha256(src.encode()).hexdigest() for nm, src in MODULES.items()}
(WORK / "library_hashes.json").write_text(json.dumps(MODULE_HASHES, indent=2))

# Purge before importing: Python caches modules in sys.modules, so re-running this cell
# after updating the notebook would silently keep the previous version of the library.
import importlib
for nm in MODULES:
    sys.modules.pop(nm[:-3], None)
importlib.invalidate_caches()
import crvs_data
REQUIRED_LIB = 4
if getattr(crvs_data, "LIB_VERSION", 0) < REQUIRED_LIB:
    raise RuntimeError(
        f"\n{'='*74}\n  Stale crvs_data: version "
        f"{getattr(crvs_data, 'LIB_VERSION', 'missing')}, need >= {REQUIRED_LIB}."
        f"\n  Restart the kernel (Run -> Restart & Run All).\n{'='*74}")
print(f"\ncrvs_data v{crvs_data.LIB_VERSION} loaded from {crvs_data.__file__}")

In [ ]:
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        raise RuntimeError("\n" + "="*74 +
            "\n  HF_TOKEN not found. Add-ons -> Secrets -> HF_TOKEN (write) -> attach.\n" + "="*74)

from crvs_sync import HFSync
sync = HFSync(repo_id=CFG["DST_REPO"], local_dir=WORK, token=HF_TOKEN, repo_type="model",
              private=CFG["HF_PRIVATE"], run_id=CFG["RUN_ID"],
              push_interval_s=CFG["PUSH_INTERVAL_S"],
              max_upload_calls_hour=CFG["HF_MAX_UPLOADS_HOUR"])
print("\nresults repo:", sync.url, "(public)")

_M = {"f": False, "n": ""}
def MAJOR(nm):
    _M["f"] = True; _M["n"] = nm
def _hook(r=None):
    if _M["f"]:
        nm = _M["n"]; _M["f"] = False; _M["n"] = ""; sync.stage_done(nm)
try:
    get_ipython().events.register("post_run_cell", _hook)
    print("post-run-cell push hook registered")
except Exception as e:
    print("hook unavailable:", e)

sync.pull(allow_patterns=["*.json", "*.jsonl", "*.csv", "*.md",
                          "runs/**/summary.json", "runs/**/state.json", "results/*"])
STATE = sync.load_state({"completed": [], "sessions": 0, "version": 2})
STATE["sessions"] = STATE.get("sessions", 0) + 1
sync.save_state(STATE)
print(f"session #{STATE['sessions']}  |  {len(STATE['completed'])} run(s) already complete")
MAJOR("00_setup")

In [ ]:
from huggingface_hub import snapshot_download
DATA = None
input_root = Path("/kaggle/input")
if input_root.exists():
    for candidate in input_root.rglob("windows.parquet"):
        if (candidate.parent / "recordings").exists() and (candidate.parent / "norm_stats.json").exists():
            DATA = candidate.parent; break
if DATA is not None:
    print("using attached Kaggle NB02 output:", DATA)
else:
    DATA = SCRATCH / "corpus"; t0 = time.time()
    print("NB02 output not attached; falling back to Hugging Face download.")
    snapshot_download(CFG["SRC_REPO"], repo_type="dataset", token=HF_TOKEN, local_dir=str(DATA),
                      allow_patterns=["recordings/*.npy", "recordings/*.json",
                                      "recordings/*.npz", "windows.parquet", "recordings.csv",
                                      "norm_stats.json", "experiments.json"], max_workers=4)
    print(f"corpus downloaded in {time.time()-t0:.0f}s")
W = pd.read_parquet(DATA / "windows.parquet")
DATA_HASH = hashlib.sha256((DATA / "windows.parquet").read_bytes()).hexdigest()
RECS = pd.read_csv(DATA / "recordings.csv")
NORM = json.loads((DATA / "norm_stats.json").read_text())
EXPINFO = json.loads((DATA / "experiments.json").read_text())
EXPERIMENTS = EXPINFO["experiments"]
REC_DIR = DATA / "recordings"
_npy = {p.stem for p in REC_DIR.glob("*.npy")}
_npz = {p.stem for p in REC_DIR.glob("*.npz")}
_have = _npy | _npz
print(f"recording files: {len(_npy)} .npy (fast) + {len(_npz)} .npz (legacy)")
if not _have:
    raise RuntimeError("No recording files downloaded -- check NB02 finished and that "
                       "SRC_REPO matches its DST_REPO.")
_missing = sorted(set(RECS["rec_id"]) - _have)
if _missing:
    print(f"WARNING: {len(_missing)} recording(s) have no file; dropping their windows")
    W = W[~W["rec_id"].isin(_missing)].reset_index(drop=True)
    RECS = RECS[~RECS["rec_id"].isin(_missing)].reset_index(drop=True)
print(f"windows {len(W):,} | recordings {len(RECS)} | subjects {RECS['subject'].nunique()}")
print("channel order on disk:", EXPINFO["channels"])

---
# 3 · Build the network and check the budget

Before training we prove the model runs, count its parameters, and confirm every head produces a
finite output with gradients. The parameter budget matters for the paper: the baseline reports
**no** parameter or FLOP count, so "we win *and* we are smaller" is a free column in our results
table — provided we actually are smaller. Target is **under 5 M**.

In [ ]:
import torch.nn.functional as F
from crvs_cmnet import build_cmnet
from crvs_models import build_baseline, count_params
from crvs_losses import CompositeLoss, MSEOnly
from crvs_engine import Trainer, seed_all, pick_device

seed_all(CFG["SEED"])
dev, ngpu, gnames = pick_device()
print(f"device {dev} | gpus {ngpu} {gnames}\n")

def make_model(spec):
    if spec["kind"] == "baseline":
        return build_baseline(spec["model"], in_ch=len(spec["channels"]), out_ch=1,
                              base=64, levels=CFG["LEVELS"])
    return build_cmnet(in_ch=len(spec["channels"]), base=CFG["BASE"], levels=CFG["LEVELS"],
                       d_ssm=CFG["D_SSM"], ssm_blocks=CFG["SSM_BLOCKS"],
                       d_state=CFG["D_STATE"], bottleneck=spec.get("bottleneck", "ssm"),
                       use_wavelet=spec.get("wavelet", True),
                       multitask=spec.get("multitask", True),
                       use_film=spec.get("film", True), dropout=CFG["DROPOUT"])

VARIANTS = {
  # kind      channels                  bottleneck   wavelet multitask film  loss
  "L2_loss_only":  dict(kind="baseline", model="multireslinknet", channels=CFG["CHANNELS_BASE"], loss="composite"),
  "L3_c1_only":    dict(kind="baseline", model="multireslinknet", channels=CFG["CHANNELS_FULL"], loss="mse"),
  "L4_c1_c5":      dict(kind="baseline", model="multireslinknet", channels=CFG["CHANNELS_FULL"], loss="composite"),
  "L5_no_wavelet": dict(kind="cmnet", channels=CFG["CHANNELS_FULL"], wavelet=False, loss="composite"),
  "L6_no_ssm":     dict(kind="cmnet", channels=CFG["CHANNELS_FULL"], bottleneck="conv", loss="composite"),
  "L7_singletask": dict(kind="cmnet", channels=CFG["CHANNELS_FULL"], multitask=False, loss="composite"),
  "L8_no_film":    dict(kind="cmnet", channels=CFG["CHANNELS_FULL"], film=False, loss="composite"),
  "L9_full":       dict(kind="cmnet", channels=CFG["CHANNELS_FULL"], loss="composite"),
  "L10_transformer": dict(kind="cmnet", channels=CFG["CHANNELS_FULL"], bottleneck="transformer", loss="composite"),
}

print(f"{'variant':<18}{'in':>4}{'params':>12}{'MB':>7}{'fwd ms':>9}  heads")
print("-" * 78)
budget = []
for nm, spec in VARIANTS.items():
    m = make_model(spec).to(dev); m.train()
    x = torch.randn(2, len(spec["channels"]), 1024, device=dev)
    t0 = time.time(); out = m(x)
    if dev.type == "cuda":
        torch.cuda.synchronize()
    dt = (time.time() - t0) * 1000
    y = torch.randn(2, 1, 1024, device=dev).clamp(-1, 1)
    pk = torch.rand(2, 1, 1024, device=dev)
    rr = torch.rand(2, 1, 1024, device=dev) + 0.5
    lf = CompositeLoss(**{f"w_{k}": v for k, v in CFG["W"].items()},
                       huber_delta=CFG["HUBER_DELTA"], peak_weight=CFG["PEAK_WEIGHT"])
    loss, parts = lf(out, y, pk, rr)
    loss.backward()
    gn = sum(float(p.grad.norm()) for p in m.parameters() if p.grad is not None)
    p = count_params(m)
    try:
        from torch.utils.flop_counter import FlopCounterMode
        with torch.no_grad(), FlopCounterMode(display=False) as fc:
            m(x[:1])
        gflops = float(fc.get_total_flops()) / 1e9
    except Exception:
        gflops = float("nan")
    heads = "+".join(k for k in ("wave", "peak", "rr") if k in out)
    ok = torch.isfinite(out["wave"]).all() and gn > 0 and math.isfinite(float(loss))
    budget.append({"variant": nm, "in_ch": len(spec["channels"]), "params": p,
                   "mb": p * 4 / 2**20, "fwd_ms_batch2": dt,
                   "gflops_per_window": gflops, "ok": bool(ok)})
    print(f"{nm:<18}{len(spec['channels']):>4}{p:>12,}{p*4/2**20:>7.1f}{dt:>9.1f}  "
          f"{heads}  {'OK' if ok else 'FAIL'}")
    del m, out, loss
    gc.collect()
    if dev.type == "cuda":
        torch.cuda.empty_cache()

B = pd.DataFrame(budget)
BUDGET_BY_VARIANT = B.set_index("variant").to_dict("index")
B.to_csv(WORK / "results" / "variant_budget.csv", index=False)
if not B["ok"].all():
    raise RuntimeError("a variant failed its smoke test:\n" + B.to_string(index=False))

# ---- structural self-checks -------------------------------------------------
# Two invariants that are invisible if they break: a decoder skip whose channels do not
# line up gets dropped silently (that once cost 59 % of MultiResLinkNet's gradient), and a
# wavelet branch one octave off its conv counterpart makes C2 fuse mismatched resolutions
# while still training fine. Both are checked here, before any GPU time is spent.
print("\nstructural self-checks")
print("-" * 66)
m = make_model(VARIANTS["L9_full"]).to(dev); m.train()
xx = torch.randn(2, 8, 1024, device=dev)
_interp = F.interpolate
_calls = {"n": 0}
def _counting(*a, **k):
    _calls["n"] += 1
    return _interp(*a, **k)
F.interpolate = _counting
try:
    oo = m(xx)
finally:
    F.interpolate = _interp
_verdict = ("aligned octave for octave" if _calls["n"] == 0
            else "MISALIGNED -- C2 would be fusing mismatched resolutions")
print(f"  fusion upsampling calls : {_calls['n']}   ({_verdict})")

with torch.no_grad():
    h0 = m.stem(m.mix(xx))
    wf = m.wave(h0)
    hh = h0; lens = []
    for i, e in enumerate(m.encs):
        hh = e(hh)
        lens.append((i, hh.shape[-1], wf[i].shape[-1], hh.shape[1], wf[i].shape[1]))
        hh = F.max_pool1d(hh, 2)
print(f"  {'level':<7}{'conv len':>10}{'wavelet len':>13}{'conv ch':>9}{'wav ch':>8}   match")
for i, cl, wl, cc, wc in lens:
    print(f"  {i:<7}{cl:>10}{wl:>13}{cc:>9}{wc:>8}   {'yes' if cl == wl else 'NO'}")
if any(cl != wl for _, cl, wl, _, _ in lens):
    raise RuntimeError("wavelet branch is not octave-aligned with the convolution branch")

oo["wave"].sum().backward()
dead = [n_ for n_, p_ in m.named_parameters()
        if p_.grad is None or float(p_.grad.abs().sum()) == 0.0]
n_dead = sum(p_.numel() for n_, p_ in m.named_parameters() if n_ in set(dead))
print(f"\n  parameters receiving no gradient: {len(dead)} tensors / {n_dead:,} values")
if dead:
    from collections import Counter
    print("  by submodule:", dict(Counter(d.split(".")[0] for d in dead)))
    raise RuntimeError(
        "some parameters get no gradient — they cost compute and learn nothing. "
        "A dropped skip connection is the usual cause.")
print("  every parameter is reachable by the loss.")
del m, oo
gc.collect()
if dev.type == "cuda":
    torch.cuda.empty_cache()
full = B[B["variant"] == "L9_full"].iloc[0]
print(f"\nCardioMamba-Net (full): {full['params']:,} params ({full['mb']:.1f} MB)")
print(f"budget target < 5 M   -> {'WITHIN BUDGET' if full['params'] < 5e6 else 'OVER BUDGET'}")
print("\nloss terms on the smoke batch:", {k: round(v, 4) for k, v in parts.items()})
MAJOR("01_smoke")

---
# 4 · Verify the composite loss actually does what it claims

A loss with six terms is easy to get wrong in a way that still trains. Two checks:

1. **Every term responds to the right error.** Corrupt a signal in a specific way and confirm the
   corresponding term rises while the others stay put — smoothing the QRS should spike the
   peak-weighted and STFT terms far more than plain Huber. That difference *is* the whole argument
   for C5, so we measure it rather than asserting it.
2. **The gradient reaches every head.** If the RR head has no gradient path, the multi-task claim
   is empty.

In [ ]:
import torch.nn.functional as Fnn
from scipy import signal as ss_

fs = 128; L = 1024
t = np.arange(L) / fs
beat = int(fs * 60 / 68)
clean = np.zeros(L); clean[::beat] = 1.0
clean = ss_.convolve(clean, ss_.windows.gaussian(21, 2.2), "same")
clean = clean / (np.abs(clean).max() + 1e-9)
pkmap = np.zeros(L)
for p in range(0, L, beat):
    a, b = max(0, p - 9), min(L, p + 10)
    g = np.exp(-0.5 * ((np.arange(a, b) - p) / 3.0) ** 2)
    pkmap[a:b] = np.maximum(pkmap[a:b], g)

def T(a):
    return torch.tensor(a, dtype=torch.float32, device=dev).reshape(1, 1, -1)

y = T(clean); pk = T(pkmap); rr = T(np.full(L, beat / fs))
cases = {
    "perfect":            clean.copy(),
    "smoothed QRS":       ss_.convolve(clean, ss_.windows.gaussian(25, 5), "same"),
    "amplitude x0.5":     clean * 0.5,
    "shifted +6 samples": np.roll(clean, 6),
    "white noise added":  clean + 0.10 * np.random.RandomState(0).randn(L),
}
lf = CompositeLoss(**{f"w_{k}": v for k, v in CFG["W"].items()},
                   huber_delta=CFG["HUBER_DELTA"], peak_weight=CFG["PEAK_WEIGHT"])
rows = []
for nm, sig in cases.items():
    sig = sig / (np.abs(sig).max() + 1e-9)
    pred = {"wave": T(sig), "peak": T(pkmap * 4 - 2), "rr": T(np.full(L, beat / fs))}
    tot, parts = lf(pred, y, pk, rr)
    mse = float(Fnn.mse_loss(T(sig), y))
    rows.append({"case": nm, "MSE": mse, **{k: round(v, 5) for k, v in parts.items()},
                 "total": round(float(tot), 5)})
D = pd.DataFrame(rows)
print(D.to_string(index=False))

base = D[D["case"] == "perfect"].iloc[0]
sm = D[D["case"] == "smoothed QRS"].iloc[0]
print("\nsmoothed QRS vs perfect -- relative rise per term:")
for term in ("MSE", "huber", "stft", "peakw", "corr"):
    if term in D.columns:
        b = max(float(base[term]), 1e-6)
        print(f"  {term:<8} x{float(sm[term])/b:>8.1f}")
print("\nThis is the argument for C5 in one table: smoothing the QRS barely moves MSE,")
print("but the peak-weighted and spectral terms react strongly. A model trained on MSE")
print("alone has almost no incentive to keep the R peak sharp.")
D.to_csv(WORK / "results" / "loss_probe.csv", index=False)

m = make_model(VARIANTS["L9_full"]).to(dev); m.train()
x = torch.randn(2, 8, 1024, device=dev)
out = m(x)
loss, _ = lf(out, torch.randn(2,1,1024,device=dev).clamp(-1,1),
             torch.rand(2,1,1024,device=dev), torch.rand(2,1,1024,device=dev)+.5)
loss.backward()
heads = {"head_wave": 0.0, "head_peak": 0.0, "head_rr": 0.0, "film": 0.0, "wave": 0.0, "bott": 0.0}
for nmp, p in m.named_parameters():
    if p.grad is None:
        continue
    for h in heads:
        if nmp.startswith(h):
            heads[h] += float(p.grad.norm())
print("\ngradient norm reaching each component:")
for h, v in heads.items():
    print(f"  {h:<12} {v:>12.5f}  {'OK' if v > 0 else 'NO GRADIENT'}")
assert heads["head_peak"] > 0 and heads["head_rr"] > 0, "a multi-task head is not learning"
del m
gc.collect()
if dev.type == "cuda":
    torch.cuda.empty_cache()
MAJOR("02_loss_probe")

---
# 5 · The ablation ladder queue

Rungs 2–10 on the headline experiment (`B_rva`), plus the full model on every other experiment so
NB05 has the per-scenario and all-five-scenario numbers. Rung 1 (MultiResLinkNet + MSE) comes from
NB03 — it is the same run, so there is no reason to train it twice.

In [ ]:
folds = list(range(CFG["QUICK_FOLDS"] if CFG["QUICK"] else CFG["N_FOLDS"]))
EPOCHS = CFG["QUICK_EPOCHS"] if CFG["QUICK"] else CFG["EPOCHS"]

QUEUE = []
queue_variants = ["L9_full"] if CFG["QUICK"] else list(VARIANTS)
for vname in queue_variants:                            # ladder on B_rva
    for f in folds:
        prefix = "quick__" if CFG["QUICK"] else ""
        QUEUE.append({"run_id": f"{prefix}{CFG['EXPERIMENT']}__{vname}__f{f}",
                      "exp": CFG["EXPERIMENT"], "variant": vname, "fold": f})
if not CFG["QUICK"]:                                    # full model everywhere else
    for exp in CFG["EXTRA_EXPERIMENTS"]:
        for f in folds:
            QUEUE.append({"run_id": f"{exp}__L9_full__f{f}", "exp": exp,
                          "variant": "L9_full", "fold": f})
    if CFG["RUN_LOSO"]:
        for lid in EXPINFO.get("loso_values", sorted(map(int, W["loso_id"].unique()))):
            QUEUE.append({"run_id": f"D_loso__L9_full__s{lid}", "exp": "D_loso",
                          "variant": "L9_full", "fold": int(lid)})
    if CFG["RUN_CROSS_SCENARIO"]:
        for sc in EXPINFO.get("cross_scenarios", sorted(W["scenario_canon"].unique())):
            safe = str(sc).lower().replace("-", "_").replace(" ", "_")
            QUEUE.append({"run_id": f"F_cross_{safe}__L9_full", "exp": f"F_cross:{sc}",
                          "variant": "L9_full", "fold": 0})

done = set(STATE.get("completed", []))
todo = [q for q in QUEUE if q["run_id"] not in done]
print(f"queue: {len(QUEUE)} run(s) | {len(done)} done | {len(todo)} remaining")
print(f"epochs {EPOCHS} | folds {folds} | budget {CFG['TIME_BUDGET_H']} h")
if CFG["QUICK"]:
    print("\n>>> QUICK MODE: full model only, 1 fold, 10 epochs. Then set QUICK=False.")
print("\nnext up:")
for q in todo[:10]:
    print("   ", q["run_id"])
if len(todo) > 10:
    print(f"    ... and {len(todo)-10} more")

In [ ]:
from crvs_data import WindowDataset, FS
from crvs_metrics import seg_metrics, detect_r_peaks, hrv_from_peaks, peak_detection_scores

def split_for(exp, fold, n_folds=None):
    n_folds = n_folds or CFG["N_FOLDS"]
    if exp == "D_loso":
        ids = EXPINFO.get("loso_values", sorted(map(int, W["loso_id"].unique())))
        fold = int(fold); vid = ids[(ids.index(fold) + 1) % len(ids)]
        sub = W[W["scenario_canon"].isin(EXPERIMENTS["C_all5"])]
        tr = sub[~sub["loso_id"].isin([fold, vid])]
        va = sub[(sub["loso_id"] == vid) & sub["no_overlap"]]
        te = sub[(sub["loso_id"] == fold) & sub["no_overlap"]]
        return tr, va, te
    if exp.startswith("F_cross:"):
        test_sc = exp.split(":", 1)[1]
        scenarios = EXPINFO.get("cross_scenarios", list(EXPERIMENTS["C_all5"]))
        val_sc = scenarios[(scenarios.index(test_sc) + 1) % len(scenarios)]
        tr = W[~W["scenario_canon"].isin([test_sc, val_sc])]
        va = W[(W["scenario_canon"] == val_sc) & W["no_overlap"]]
        te = W[(W["scenario_canon"] == test_sc) & W["no_overlap"]]
        return tr, va, te
    sub = W[W["scenario_canon"].isin(EXPERIMENTS[exp])]
    te_g, va_g = fold % n_folds, (fold + 1) % n_folds
    tr = sub[~sub["fold_group"].isin([te_g, va_g])]
    va = sub[(sub["fold_group"] == va_g) & sub["no_overlap"]]
    te = sub[(sub["fold_group"] == te_g) & sub["no_overlap"]]
    assert not (set(tr["subject"]) & set(te["subject"])), "SUBJECT LEAK"
    return tr, va, te

def make_datasets(exp, fold, channels):
    tr, va, te = split_for(exp, fold)
    if not exp.startswith("F_cross:"):
        assert not (set(tr["subject"]) & set(te["subject"])), "SUBJECT LEAK train/test"
        assert not (set(va["subject"]) & set(te["subject"])), "SUBJECT LEAK val/test"
    norm_key = f"{exp}|{fold}" if not exp.startswith("F_cross:") else f"{exp}|0"
    norm = NORM.get(norm_key)
    if norm is None:
        raise RuntimeError(f"no normalisation stats for {norm_key} -- re-run NB02 v2")
    idx = [EXPINFO["channels"].index(c) for c in channels]
    sn = {"mean": [norm["mean"][i] for i in idx], "std": [norm["std"][i] for i in idx]}
    mk = lambda d, aug: WindowDataset(REC_DIR, d, sn, channels, augment=aug,
                                      seed=CFG["SEED"] + fold)
    return mk(tr, True), mk(va, False), mk(te, False), (tr, va, te)

def evaluate(Y, P, index, out_dir):
    idx_frame = index.reset_index(drop=True)
    subs = idx_frame["subject"].to_numpy(); scen = idx_frame["scenario_canon"].to_numpy()
    rows = []
    for i in range(len(Y)):
        m = seg_metrics(Y[i], P[i], FS)
        m["subject"] = subs[i] if i < len(subs) else "?"
        m["scenario"] = scen[i] if i < len(scen) else "?"
        rows.append(m)
    dfw = pd.DataFrame(rows)
    dfw.to_parquet(out_dir / "metrics_windows.parquet", index=False)
    num = [c for c in dfw.columns if dfw[c].dtype.kind in "fi"]
    agg = {c: float(dfw[c].mean()) for c in num}
    agg.update({c + "_std": float(dfw[c].std()) for c in num})
    hr = []
    for rid, grp in idx_frame.assign(_row=np.arange(len(idx_frame))).groupby("rec_id"):
        grp = grp.sort_values("start"); pos = grp["_row"].to_numpy()
        if len(pos) < 2:
            continue
        yg = np.concatenate(Y[pos]); yp = np.concatenate(P[pos])
        g = hrv_from_peaks(detect_r_peaks(yg, FS), FS)
        pr = hrv_from_peaks(detect_r_peaks(yp, FS), FS)
        hr.append({"rec_id": rid, "subject": str(grp["subject"].iloc[0]),
                   "scenario": str(grp["scenario_canon"].iloc[0]),
                   **{f"gt_{k}": v for k, v in g.items()},
                   **{f"pr_{k}": v for k, v in pr.items()},
                   **peak_detection_scores(yg, yp, FS)})
    dfh = pd.DataFrame(hr)
    if len(dfh):
        dfh.to_parquet(out_dir / "metrics_recordings.parquet", index=False)
        numeric = [c for c in dfh.columns if dfh[c].dtype.kind in "fi"]
        dfh.groupby("subject", as_index=False)[numeric].mean().to_parquet(
            out_dir / "metrics_subjects.parquet", index=False)
        for k in ("F1", "precision", "recall", "accuracy", "missed_rate", "timing_err_ms_median"):
            if k in dfh.columns:
                agg["peak_" + k] = float(dfh[k].mean())
        for k in ("mean_hr_bpm", "rmssd_ms"):
            if f"gt_{k}" in dfh and f"pr_{k}" in dfh:
                agg["MAE_" + k] = float((dfh[f"gt_{k}"] - dfh[f"pr_{k}"]).abs().mean())
    return agg, dfw

---
# 6 · Train the ladder

Interrupt-safe and resumable at the epoch level, exactly as in NB03. Watch `CC_t` in the summary
line after each run — the rungs should climb: rung 2 (loss only) should already beat the NB03
baseline, and rung 9 (full) should be the best of the ladder. If rung 9 is *not* the best, the
ablation is telling you something real and the paper should report it honestly rather than the
architecture being quietly retuned until it wins.

In [ ]:
t_start = time.time(); budget_s = CFG["TIME_BUDGET_H"] * 3600
completed_now = []

for qi, q in enumerate(todo, 1):
    el = time.time() - t_start
    if el > budget_s:
        print(f"\n=== time budget reached ({el/3600:.2f} h). Stopping cleanly. ===")
        print(f"    {len(todo)-qi+1} run(s) left -- start a new session and re-run.")
        break
    rid, exp, vname, fold = q["run_id"], q["exp"], q["variant"], q["fold"]
    spec = VARIANTS[vname]
    out = WORK / "runs" / rid; out.mkdir(parents=True, exist_ok=True)
    print("\n" + "=" * 78)
    print(f"[{qi}/{len(todo)}]  {rid}   ({el/3600:.2f} h elapsed)")
    print("=" * 78)
    try:
        if (out / "state.json").exists() and not (out / "state.pt").exists():
            print("  interrupted remote run found; restoring exact checkpoint...")
            sync.pull(allow_patterns=[f"runs/{rid}/state.pt", f"runs/{rid}/best.pt",
                                      f"runs/{rid}/state.json", f"runs/{rid}/run_config.json",
                                      f"runs/{rid}/environment.json", f"runs/{rid}/*.jsonl",
                                      f"runs/{rid}/*.csv", f"runs/{rid}/validation_windows/*",
                                      f"runs/{rid}/validation_recordings/*"])
        tr_ds, va_ds, te_ds, (tri, vai, tei) = make_datasets(exp, fold, spec["channels"])
        print(f"  in_ch {len(spec['channels'])} | train {len(tr_ds):,} val {len(va_ds):,} "
              f"test {len(te_ds):,} | test subjects {sorted(tei['subject'].unique())}")
        seed_all(CFG["SEED"] + fold)
        model = make_model(spec)
        loss_fn = (CompositeLoss(**{f"w_{k}": v for k, v in CFG["W"].items()},
                                 huber_delta=CFG["HUBER_DELTA"],
                                 peak_weight=CFG["PEAK_WEIGHT"])
                   if spec["loss"] == "composite" else MSEOnly())
        tr = Trainer(model, loss_fn, out, rid, sync=sync, lr=CFG["LR"],
                     weight_decay=CFG["WEIGHT_DECAY"], epochs=EPOCHS,
                     patience=CFG["PATIENCE"], batch_size=CFG["BATCH"],
                     num_workers=CFG["WORKERS"], amp=CFG["AMP"], multi_gpu=CFG["MULTI_GPU"],
                     log_every=CFG["LOG_EVERY"],
                     checkpoint_every_steps=CFG["CHECKPOINT_EVERY_STEPS"],
                     checkpoint_every_s=CFG["CHECKPOINT_EVERY_S"], seed=CFG["SEED"] + fold,
                     require_dual_gpu=CFG["REQUIRE_DUAL_T4"],
                     run_config={"experiment": exp, "variant": vname, "fold": fold,
                                 "epochs": EPOCHS, "spec": spec, "base": CFG["BASE"],
                                 "d_ssm": CFG["D_SSM"], "ssm_blocks": CFG["SSM_BLOCKS"],
                                 "d_state": CFG["D_STATE"], "levels": CFG["LEVELS"],
                                 "dropout": CFG["DROPOUT"], "loss_weights": CFG["W"],
                                 "lr": CFG["LR"], "batch": CFG["BATCH"],
                                 "seed": CFG["SEED"] + fold,
                                 "data_index_sha256": DATA_HASH, "library_sha256": MODULE_HASHES,
                                 "train_subjects": sorted(map(str, tri["subject"].unique())),
                                 "val_subjects": sorted(map(str, vai["subject"].unique())),
                                 "test_subjects": sorted(map(str, tei["subject"].unique()))})
        tr.load(); tr.fit(tr_ds, va_ds)
        Y, P = tr.predict(te_ds)
        agg, dfw = evaluate(Y, P, tei, out)
        keep = min(200, len(Y)); sel = np.linspace(0, len(Y) - 1, keep).astype(int)
        np.savez_compressed(out / "preds_sample.npz", y=Y[sel].astype(np.float32),
                            p=P[sel].astype(np.float32),
                            subject=tei["subject"].to_numpy()[sel].astype(str))
        (out / "summary.json").write_text(json.dumps({
            "run_id": rid, "experiment": exp, "variant": vname, "fold": fold,
            "spec": {k: v for k, v in spec.items()},
            "params": count_params(tr.raw_model), "epochs_run": tr.state["epoch"],
            "gflops_per_window": BUDGET_BY_VARIANT.get(vname, {}).get("gflops_per_window"),
            "forward_ms_batch2": BUDGET_BY_VARIANT.get(vname, {}).get("fwd_ms_batch2"),
            "best_epoch": tr.state["best_epoch"], "best_val": tr.state["best"],
            "n_train": len(tr_ds), "n_val": len(va_ds), "n_test": len(te_ds),
            "test_subjects": sorted(map(str, tei["subject"].unique())),
            "metrics": agg, "finished_utc": datetime.now(timezone.utc).isoformat()},
            indent=2, default=str))
        print(f"  --> CC_t {agg['CC_temporal']:.2f}  CC_s {agg['CC_spectral']:.2f}  "
              f"MAE {agg['MAE']:.5f}  RRMSE_t {agg['RRMSE_temporal']:.4f}  "
              f"F1 {agg.get('peak_F1', float('nan')):.3f}  "
              f"dRMSSD {agg.get('MAE_rmssd_ms', float('nan')):.1f} ms")
        done.add(rid); completed_now.append(rid)
        STATE["completed"] = sorted(done); sync.save_state(STATE)
        pushed = sync.flush(force=True, msg=f"{rid} complete CCt={agg['CC_temporal']:.2f}")
        if pushed:
            for name in ("state.pt", "best.pt"):
                p = out / name
                if p.exists(): p.unlink()
            sync.log("local_checkpoints_pruned", run_id=rid, remote_copy=True)
        del tr, model, tr_ds, va_ds, te_ds, Y, P
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except KeyboardInterrupt:
        print("\ninterrupted -- checkpoint saved and pushed; re-run to resume this run.")
        raise
    except Exception as e:
        import traceback
        print(f"  !! {type(e).__name__}: {e}")
        (out / "error.txt").write_text(traceback.format_exc())
        sync.log("run_failed", run=rid, err=f"{type(e).__name__}: {e}")

print(f"\ncompleted this session: {len(completed_now)}  |  total {len(done)}/{len(QUEUE)}")
MAJOR("03_training")

---
# 7 · The ablation table

The table the paper's Discussion is built on. Each rung adds one component; the `Δ CC_t` column is
what that component is worth. This is what turns "our architecture is better" into "the SSM
bottleneck contributes 3.2 points of temporal correlation".

Rung 1 is pulled from NB03's separate baseline v2 repo.

In [ ]:
rows = []
for p in sorted((WORK / "runs").glob("*/summary.json")):
    try:
        s = json.loads(p.read_text())
        rows.append({"experiment": s["experiment"], "variant": s.get("variant", s.get("model")),
                     "fold": s["fold"], "params": s.get("params"),
                     **{k: v for k, v in s["metrics"].items() if not k.endswith("_std")}})
    except Exception:
        pass
# Rung 1 comes from NB03's separate v2 repo. Pull summaries only (not checkpoints).
baseline_cache = SCRATCH / "baseline_compare"
try:
    snapshot_download(CFG["BASELINE_REPO"], repo_type="model", token=HF_TOKEN,
                      local_dir=str(baseline_cache),
                      allow_patterns=["runs/B_rva__multireslinknet__f*/summary.json"], max_workers=4)
except Exception as e:
    print("baseline summaries unavailable:", type(e).__name__, e)
for p in sorted((baseline_cache / "runs").glob("B_rva__multireslinknet__f*/summary.json")):
    try:
        s = json.loads(p.read_text())
        rows.append({"experiment": s["experiment"], "variant": "L1_baseline_mse",
                     "fold": s["fold"], "params": s.get("params"),
                     **{k: v for k, v in s["metrics"].items() if not k.endswith("_std")}})
    except Exception:
        pass

R = pd.DataFrame(rows)
if not len(R):
    print("no completed runs yet -- run the training cell.")
else:
    R.to_csv(WORK / "results" / "runs_raw.csv", index=False)
    LADDER = ["L1_baseline_mse", "L2_loss_only", "L3_c1_only", "L4_c1_c5", "L5_no_wavelet",
              "L6_no_ssm", "L7_singletask", "L8_no_film", "L9_full", "L10_transformer"]
    LABEL = {
      "L1_baseline_mse": "1. MultiResLinkNet + MSE  (baseline)",
      "L2_loss_only":    "2. + composite loss (C5)",
      "L3_c1_only":      "3. + 8-channel input (C1)",
      "L4_c1_c5":        "4. + C1 + C5",
      "L5_no_wavelet":   "5. CardioMamba, no wavelet (-C2)",
      "L6_no_ssm":       "6. CardioMamba, no SSM (-C3)",
      "L7_singletask":   "7. CardioMamba, single-task (-C4)",
      "L8_no_film":      "8. CardioMamba, no FiLM refinement",
      "L9_full":         "9. CardioMamba-Net (full, C1-C5)",
      "L10_transformer": "10. Transformer bottleneck (control)",
    }
    cols = ["MAE", "MSE", "CC_temporal", "CC_spectral", "RRMSE_temporal", "RRMSE_spectral"]
    extra = [c for c in ("peak_F1", "MAE_mean_hr_bpm", "MAE_rmssd_ms") if c in R.columns]
    b = R[R["experiment"] == CFG["EXPERIMENT"]]
    A = (b.groupby("variant")[cols + extra + ["params"]].mean()
          .reindex([v for v in LADDER if v in set(b["variant"])]))
    A["folds"] = b.groupby("variant").size().reindex(A.index)
    ref = A.loc["L1_baseline_mse", "CC_temporal"] if "L1_baseline_mse" in A.index else np.nan
    A["dCC_t_vs_baseline"] = (A["CC_temporal"] - ref).round(2)
    A.index = [LABEL.get(i, i) for i in A.index]
    pd.set_option("display.width", 220, "display.max_columns", 40)
    print("=" * 118)
    print(f"ABLATION LADDER  —  experiment {CFG['EXPERIMENT']}  (mean over folds)")
    print("=" * 118)
    print(A.round(5).to_string())
    A.to_csv(WORK / "results" / "ablation.csv")

    print("\n" + "-" * 118)
    print("TARGETS FROM PLAN.md")
    print("-" * 118)
    if "9. CardioMamba-Net (full, C1-C5)" in A.index:
        f = A.loc["9. CardioMamba-Net (full, C1-C5)"]
        checks = [("CC_temporal >= 80", f["CC_temporal"], 80, "ge"),
                  ("CC_spectral >= 88", f["CC_spectral"], 88, "ge"),
                  ("params < 5 M", f["params"], 5e6, "lt")]
        if "MAE_mean_hr_bpm" in A.columns:
            checks.append(("HR MAE < 2 bpm", f["MAE_mean_hr_bpm"], 2, "lt"))
        if "MAE_rmssd_ms" in A.columns:
            checks.append(("RMSSD MAE < 8 ms", f["MAE_rmssd_ms"], 8, "lt"))
        for nm, v, thr, op in checks:
            hit = (v >= thr) if op == "ge" else (v < thr)
            print(f"  {nm:<22} actual {v:>12,.3f}   {'MET' if hit else 'not yet'}")
        print(f"\n  vs the published MultiResLinkNet (CC_t 61.86, CC_s 79.96):")
        print(f"    CC_temporal {f['CC_temporal']:.2f}  ({f['CC_temporal']-61.86:+.2f})")
        print(f"    CC_spectral {f['CC_spectral']:.2f}  ({f['CC_spectral']-79.96:+.2f})")
MAJOR("04_ablation")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
S = {"radar": "#0F7C82", "ecg": "#AF3A2C", "muted": "#5C6B71", "ink": "#10171B",
     "grid": "#D3DADB", "amber": "#8A6212"}
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 160, "savefig.bbox": "tight",
                     "axes.grid": True, "grid.color": S["grid"], "grid.linewidth": .6,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 8.5, "axes.titlesize": 10, "axes.titleweight": "bold"})
FIG = WORK / "figures"

if len(R):
    b = R[R["experiment"] == CFG["EXPERIMENT"]]
    order = [v for v in LADDER if v in set(b["variant"])]
    if order:
        vals = [b[b["variant"] == v]["CC_temporal"].mean() for v in order]
        cols_ = [S["ecg"] if v == "L9_full" else
                 S["muted"] if v == "L1_baseline_mse" else S["radar"] for v in order]
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.barh(range(len(order)), vals, color=cols_, edgecolor="white")
        ax.set_yticks(range(len(order)))
        ax.set_yticklabels([LABEL.get(v, v) for v in order], fontsize=7.5)
        ax.invert_yaxis()
        ax.axvline(61.86, color=S["ink"], ls="--", lw=1.2, label="published MultiResLinkNet")
        for i, v in enumerate(vals):
            ax.text(v + .3, i, f"{v:.1f}", va="center", fontsize=7)
        ax.set_xlabel("temporal correlation (x100)")
        ax.set_title(f"Ablation ladder — {CFG['EXPERIMENT']}", loc="left")
        ax.legend(frameon=False, fontsize=7)
        fig.savefig(FIG / "nb04_fig1_ablation.png"); plt.close(fig)
        print("  wrote nb04_fig1_ablation.png")

    fig, ax = plt.subplots(figsize=(9, 3.4))
    for v in order:
        hs = []
        for p in (WORK / "runs").glob(f"*__{v}__*/state.json"):
            try:
                hs += [(h["epoch"], h["val"]) for h in json.loads(p.read_text())["history"]]
            except Exception:
                pass
        if hs:
            d = pd.DataFrame(hs, columns=["e", "v"]).groupby("e")["v"].mean()
            ax.plot(d.index, d.values, lw=1.2,
                    label=v, color=S["ecg"] if v == "L9_full" else None,
                    zorder=3 if v == "L9_full" else 2)
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("validation loss")
    ax.set_title("Ablation training curves", loc="left")
    ax.legend(frameon=False, ncol=3, fontsize=6.5)
    fig.savefig(FIG / "nb04_fig2_curves.png"); plt.close(fig)
    print("  wrote nb04_fig2_curves.png")

    fullp = sorted((WORK / "runs").glob(f"{CFG['EXPERIMENT']}__L9_full__f*/preds_sample.npz"))
    basep = sorted(Path(CFG["WORK"]).parent.glob(
        "nb03/runs/B_rva__multireslinknet__f*/preds_sample.npz"))
    if fullp:
        zf = np.load(fullp[0]); zb = np.load(basep[0]) if basep else None
        k = min(4, len(zf["y"]) - 1); tt = np.arange(1024) / 128.0
        rows_ = 3 if zb is not None else 2
        fig, axes = plt.subplots(rows_, 1, figsize=(11, 2.0 * rows_), sharex=True)
        axes[0].plot(tt, zf["y"][k], lw=1.1, color=S["ink"])
        axes[0].set_ylabel("ground truth", rotation=0, ha="right", va="center", fontsize=8)
        axes[1].plot(tt, zf["p"][k], lw=1.1, color=S["ecg"])
        axes[1].set_ylabel("CardioMamba-Net", rotation=0, ha="right", va="center",
                           fontsize=8, color=S["ecg"])
        if zb is not None:
            kb = min(k, len(zb["y"]) - 1)
            axes[2].plot(tt, zb["p"][kb], lw=1.1, color=S["muted"])
            axes[2].set_ylabel("MultiResLinkNet", rotation=0, ha="right", va="center", fontsize=8)
        for a in axes:
            a.tick_params(labelleft=False)
        axes[-1].set_xlabel("seconds")
        axes[0].set_title("Held-out reconstruction — is the QRS sharp, or smeared?", loc="left")
        fig.tight_layout(); fig.savefig(FIG / "nb04_fig3_qualitative.png"); plt.close(fig)
        print("  wrote nb04_fig3_qualitative.png")
MAJOR("05_figures")

In [ ]:
ok = sync.flush(final=True, msg=f"{CFG['RUN_ID']} — {len(done)}/{len(QUEUE)} runs complete")
print("\n" + "=" * 76)
print("  SESSION COMPLETE" if ok else "  SESSION COMPLETE (final push had a problem)")
print("=" * 76)
print(f"  repo      : {sync.url}")
print(f"  runs done : {len(done)}/{len(QUEUE)}   this session: {len(completed_now)}")
print(f"  elapsed   : {(time.time()-t_start)/3600:.2f} h")
print("=" * 76)
if len(done) < len(QUEUE):
    print(f"\n  {len(QUEUE)-len(done)} run(s) remain. Start a NEW session and re-run this")
    print("  notebook -- it resumes from Hugging Face and skips finished runs.")
else:
    print("\n  Ladder complete. Next: 05_evaluate_and_figures.ipynb")

---
# 8 · Troubleshooting

**CUDA out of memory** — lower `CFG["BATCH"]` to 32 or 24. The wavelet branch roughly doubles
encoder activations. If it persists, drop `CFG["D_SSM"]` to 192.

**S4D produces NaN** — the kernel is computed in float32 outside autocast on purpose. If NaNs
still appear, lower `CFG["LR"]` to 5e-4 and check `nb04_fig2_curves.png` for the epoch it began.

**`L10_transformer` much slower than `L9_full`** — expected. Attention is quadratic in sequence
length; the SSM is linear. That gap is itself a result worth reporting.

**The full model is not the best rung** — do not quietly retune until it wins. Report what the
ablation says. If `L6_no_ssm` beats `L9_full`, the SSM is not earning its place on this dataset
and the paper is more interesting for saying so.

**Param count over 5 M** — reduce `CFG["BASE"]` to 24 or `CFG["SSM_BLOCKS"]` to 2, then re-run the
smoke cell before training.

**Session ended mid-queue** — expected and handled. New session, run again, it resumes.